# Physical Climate Risk Index — US, Full Sample (2008–2025)

This notebook contains **only the code that actually ran** to produce the reported
PCRA index verified by tracing every function call from the final production
cells back rather than including everything that was ever written
or tested. Smoke tests, superseded drafts, and unused reference implementations have
been removed.


## Reproducibility Notes

1. **API key** — set `MEDIA_CLOUD_API_KEY` as an environment variable before running.
2. **Output paths** — controlled by `THESIS_DATA_ROOT` env var (defaults to `./thesis_data`).
3. **Run order** — top to bottom. Sections 1–9 are function/config definitions; Section 10
   is the actual multi-day production run.
4. **`RUN_DIRS`** in the Stage A merge cell (Section 10) must be edited to point at your
   own three Stage A output folders.
5. **Known methodology note (see Section 5):** the article-inclusion and country-attribution
   logic actually executed by `gather_corpus()` differs from a simpler, separately-documented
   version that exists elsewhere in the original pipeline notebook (single-hit hazard
   inclusion, sentence-count country attribution). Only the logic that **actually ran** is
   included here — see the note in Section 5 for the exact rule.


## Setup

In [1]:
# Consolidated dependencies — run once, restart kernel, then continue.
# %%capture
#!pip install "numpy<2"
#!pip install mediacloud trafilatura newspaper3k spacy
#!pip install "torch>=2.4" --index-url https://download.pytorch.org/whl/cpu
#!pip install "transformers>=4.41" "huggingface_hub>=0.23,<1.0" "datasets<3.0" \
#             "sentence-transformers>=3.0"
#!pip install "statsmodels>=0.14" "scipy>=1.11" scikit-learn
#!pip install pandas numpy matplotlib seaborn openpyxl
#!python -m spacy download en_core_web_trf


**Imports & global config.** 

Also sets `THESIS_DATA_ROOT` (portable project root) and the random seed.

In [2]:
from __future__ import annotations

import os, re, json, csv, time, logging, hashlib, warnings
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, date, time as dt_time, timedelta, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import cosine as cos_dist

import spacy                          # Phase 1 — NER, lemmatisation, POS
import statsmodels.api as sm          # Phase 3/4 — AR(p), local projections
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt
import seaborn as sns

import requests

warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# reproducibility 
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# project root 
# project root (portable — override via THESIS_DATA_ROOT env var) 
THESIS_DATA_ROOT = Path(os.environ.get("THESIS_DATA_ROOT", "./thesis_data")).resolve()
ROOT = THESIS_DATA_ROOT
ROOT.mkdir(parents=True, exist_ok=True)

print(f"✓ THESIS_DATA_ROOT: {THESIS_DATA_ROOT}")

# Portable path for the Actual Run Data folder 
ACTUAL_RUN_ROOT = Path(os.environ.get(
    "ACTUAL_RUN_ROOT",
    THESIS_DATA_ROOT.parent.parent  # <-- adjust this fallback if needed
))

print(f"✓ ACTUAL_RUN_ROOT: {ACTUAL_RUN_ROOT}")

print("Imports OK")
print(f"numpy: {np.__version__}")
print(f"spacy: {spacy.__version__}")


✓ THESIS_DATA_ROOT: /Users/adam/Desktop/PCRA Index Thesis/Data Gathering/thesis_data
✓ ACTUAL_RUN_ROOT: /Users/adam/Desktop/PCRA Index Thesis
Imports OK
numpy: 2.4.6
spacy: 3.8.13


## API Key

Hard coded for ease of access

In [3]:
MEDIA_CLOUD_API_KEY = "de4395d4ff5dc9b5b1adfe695ccf5219b6e2b4c1" 


## Lexicons & Configuration

**Physical-risk taxonomy (lexicon L)** from IPCC AR6/TCFD, each tagged with category, severity, Acute/Chronic/Both.

In [4]:
TAXONOMY = [
    # HYDROLOGICAL ── Flooding (Acute)
    ("Hydrological", "Flooding",         "flood",                          "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "floods",                         "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "flooding",                       "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "flash flood",                    "Acute",   3, "IPCC"),
    ("Hydrological", "Flooding",         "flash floods",                   "Acute",   3, "IPCC"),
    ("Hydrological", "Flooding",         "flash flooding",                 "Acute",   3, "IPCC"),
    ("Hydrological", "Flooding",         "coastal flood",                  "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "coastal floods",                 "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "coastal flooding",               "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "storm surge",                    "Acute",   3, "IPCC"),
    ("Hydrological", "Flooding",         "storm surges",                   "Acute",   3, "IPCC"),
    ("Hydrological", "Flooding",         "glacial lake outburst flood",    "Acute",   3, "IPCC"),
    ("Hydrological", "Flooding",         "glacial lake outburst floods",   "Acute",   3, "IPCC"),
    ("Hydrological", "Flooding",         "GLOF",                           "Acute",   3, "IPCC"),
    ("Hydrological", "Flooding",         "GLOFs",                          "Acute",   3, "IPCC"),
    ("Hydrological", "Flooding",         "groundwater flooding",           "Acute",   1, "IPCC"),
    ("Hydrological", "Flooding",         "riverine flood",                 "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "riverine floods",                "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "riverine flooding",              "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "flood damage",                   "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "pluvial flood",                  "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "pluvial floods",                 "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "pluvial flooding",               "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "surface water flood",            "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "surface water floods",           "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "surface water flooding",         "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "urban flood",                    "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "urban floods",                   "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "urban flooding",                 "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "river flood",                    "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "river floods",                   "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "river flooding",                 "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "fluvial flood",                  "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "fluvial floods",                 "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "fluvial flooding",               "Acute",   2, "IPCC"),
    ("Hydrological", "Flooding",         "flood warning",                  "Acute",   1, "IPCC"),
    ("Hydrological", "Flooding",         "flood warnings",                 "Acute",   1, "IPCC"),
    ("Hydrological", "Flooding",         "flood alert",                    "Acute",   1, "IPCC"),
    ("Hydrological", "Flooding",         "flood alerts",                   "Acute",   1, "IPCC"),

    #  METEOROLOGICAL ── Storm (Acute) 
    ("Meteorological", "Storm",          "tropical cyclone",               "Acute",   3, "IPCC/TCFD"),
    ("Meteorological", "Storm",          "tropical cyclones",              "Acute",   3, "IPCC/TCFD"),
    ("Meteorological", "Storm",          "hurricane",                      "Acute",   3, "IPCC"),
    ("Meteorological", "Storm",          "hurricanes",                     "Acute",   3, "IPCC"),
    ("Meteorological", "Storm",          "typhoon",                        "Acute",   3, "IPCC"),
    ("Meteorological", "Storm",          "typhoons",                       "Acute",   3, "IPCC"),
    ("Meteorological", "Storm",          "windstorm",                      "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "windstorms",                     "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "hailstorm",                      "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "hailstorms",                     "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "hail",                           "Acute",   1, "IPCC AR6 CID"),
    ("Meteorological", "Storm",          "lightning strike",               "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "lightning strikes",              "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "tornado",                        "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "tornadoes",                      "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "tornados",                       "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "dust storm",                     "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "dust storms",                    "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "sandstorm",                      "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "sandstorms",                     "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "storm",                          "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "storms",                         "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "severe storm",                   "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "severe storms",                  "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "thunderstorm",                   "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "thunderstorms",                  "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "severe thunderstorm",            "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "severe thunderstorms",           "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "convective storm",               "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "convective storms",              "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "extratropical cyclone",          "Acute",   3, "IPCC"),
    ("Meteorological", "Storm",          "extratropical cyclones",         "Acute",   3, "IPCC"),
    ("Meteorological", "Storm",          "post-tropical cyclone",          "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "post-tropical cyclones",         "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "cyclone",                        "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "cyclones",                       "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "storm damage",                   "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "storm warning",                  "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "storm warnings",                 "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "weather warning",                "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "weather warnings",               "Acute",   1, "IPCC"),
    ("Meteorological", "Storm",          "severe weather warning",         "Acute",   2, "IPCC"),
    ("Meteorological", "Storm",          "severe weather warnings",        "Acute",   2, "IPCC"),
    # New AR6-era additions
    ("Meteorological", "Storm",          "derecho",                        "Acute",   3, "IPCC/NOAA"),
    ("Meteorological", "Storm",          "derechos",                       "Acute",   3, "IPCC/NOAA"),
    ("Meteorological", "Storm",          "medicane",                       "Acute",   3, "IPCC"),
    ("Meteorological", "Storm",          "medicanes",                      "Acute",   3, "IPCC"),
    ("Meteorological", "Storm",          "sting jet",                      "Acute",   2, "Met Office/IPCC"),
    ("Meteorological", "Storm",          "sting jets",                     "Acute",   2, "Met Office/IPCC"),

    #  METEOROLOGICAL ── Wind (Acute) 
    ("Meteorological", "Wind",           "gale",                           "Acute",   1, "IPCC"),
    ("Meteorological", "Wind",           "gales",                          "Acute",   1, "IPCC"),
    ("Meteorological", "Wind",           "severe gale",                    "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "severe gales",                   "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "strong wind",                    "Acute",   1, "IPCC"),
    ("Meteorological", "Wind",           "strong winds",                   "Acute",   1, "IPCC"),
    ("Meteorological", "Wind",           "high wind",                      "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "high winds",                     "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "extreme wind",                   "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "extreme winds",                  "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "severe wind",                    "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "severe winds",                   "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "damaging wind",                  "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "damaging winds",                 "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "wind gust",                      "Acute",   1, "IPCC"),
    ("Meteorological", "Wind",           "wind gusts",                     "Acute",   1, "IPCC"),
    ("Meteorological", "Wind",           "violent wind",                   "Acute",   2, "IPCC"),
    ("Meteorological", "Wind",           "violent winds",                  "Acute",   2, "IPCC"),

    #  METEOROLOGICAL ── Precipitation (Acute) 
    ("Meteorological", "Precipitation",  "extreme rainfall",               "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "snowstorm",                      "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "snowstorms",                     "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "blizzard",                       "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "blizzards",                      "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "ice storm",                      "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "ice storms",                     "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "heavy precipitation",            "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "heavy rainfall",                 "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "heavy snow",                     "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "rainfall",                       "Acute",   1, "IPCC"),
    ("Meteorological", "Precipitation",  "heavy rain",                     "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "torrential rain",                "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "torrential rainfall",            "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "intense rainfall",               "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "extreme precipitation",          "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "record rainfall",                "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "rainstorm",                      "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "rainstorms",                     "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "downpour",                       "Acute",   1, "IPCC"),
    ("Meteorological", "Precipitation",  "downpours",                      "Acute",   1, "IPCC"),
    ("Meteorological", "Precipitation",  "cloudburst",                     "Acute",   2, "IPCC"),
    ("Meteorological", "Precipitation",  "cloudbursts",                    "Acute",   2, "IPCC"),
    # AR6-era addition
    ("Meteorological", "Precipitation",  "atmospheric river",              "Acute",   3, "IPCC/AMS"),
    ("Meteorological", "Precipitation",  "atmospheric rivers",             "Acute",   3, "IPCC/AMS"),

    #  METEOROLOGICAL ── Snow and Ice (Acute) 
    ("Meteorological", "Snow and Ice",   "snow",                           "Acute",   1, "IPCC"),
    ("Meteorological", "Snow and Ice",   "snowfall",                       "Acute",   1, "IPCC"),
    ("Meteorological", "Snow and Ice",   "heavy snowfall",                 "Acute",   2, "IPCC"),
    ("Meteorological", "Snow and Ice",   "extreme snowfall",               "Acute",   2, "IPCC"),
    ("Meteorological", "Snow and Ice",   "snow accumulation",              "Acute",   1, "IPCC"),
    ("Meteorological", "Snow and Ice",   "ice accumulation",               "Acute",   1, "IPCC"),
    ("Meteorological", "Snow and Ice",   "freezing rain",                  "Acute",   2, "IPCC"),
    ("Meteorological", "Snow and Ice",   "sleet",                          "Acute",   1, "IPCC"),
    ("Meteorological", "Snow and Ice",   "wintry weather",                 "Acute",   1, "IPCC"),
    ("Meteorological", "Snow and Ice",   "winter storm",                   "Acute",   2, "IPCC"),
    ("Meteorological", "Snow and Ice",   "winter storms",                  "Acute",   2, "IPCC"),
    ("Meteorological", "Snow and Ice",   "severe winter storm",            "Acute",   3, "IPCC"),
    ("Meteorological", "Snow and Ice",   "severe winter storms",           "Acute",   3, "IPCC"),
    ("Meteorological", "Snow and Ice",   "lake effect snow",               "Acute",   2, "IPCC"),
    ("Meteorological", "Snow and Ice",   "ice jam",                        "Acute",   1, "IPCC"),
    ("Meteorological", "Snow and Ice",   "ice jams",                       "Acute",   1, "IPCC"),
    ("Meteorological", "Snow and Ice",   "black ice",                      "Acute",   1, "IPCC"),
    ("Meteorological", "Snow and Ice",   "icy conditions",                 "Acute",   1, "IPCC"),

    #  METEOROLOGICAL ── Temperature (Acute) 
    ("Meteorological", "Temperature",    "heatwave",                       "Acute",   3, "IPCC"),
    ("Meteorological", "Temperature",    "heatwaves",                      "Acute",   3, "IPCC"),
    ("Meteorological", "Temperature",    "heat wave",                      "Acute",   3, "IPCC"),
    ("Meteorological", "Temperature",    "heat waves",                     "Acute",   3, "IPCC"),
    ("Meteorological", "Temperature",    "extreme heatwave",               "Acute",   3, "IPCC"),
    ("Meteorological", "Temperature",    "extreme heatwaves",              "Acute",   3, "IPCC"),
    ("Meteorological", "Temperature",    "extreme heat wave",              "Acute",   3, "IPCC"),
    ("Meteorological", "Temperature",    "extreme heat waves",             "Acute",   3, "IPCC"),
    ("Meteorological", "Temperature",    "cold wave",                      "Acute",   2, "IPCC"),
    ("Meteorological", "Temperature",    "cold waves",                     "Acute",   2, "IPCC"),
    ("Meteorological", "Temperature",    "extreme cold",                   "Acute",   2, "IPCC"),
    ("Meteorological", "Temperature",    "frost",                          "Acute",   1, "IPCC"),
    ("Meteorological", "Temperature",    "frosts",                         "Acute",   1, "IPCC"),
    ("Meteorological", "Temperature",    "heat alert",                     "Acute",   1, "IPCC"),
    ("Meteorological", "Temperature",    "heat alerts",                    "Acute",   1, "IPCC"),
    ("Meteorological", "Temperature",    "heat warning",                   "Acute",   1, "IPCC"),
    ("Meteorological", "Temperature",    "heat warnings",                  "Acute",   1, "IPCC"),
    ("Meteorological", "Temperature",    "extreme temperature",            "Acute",   2, "IPCC"),
    ("Meteorological", "Temperature",    "extreme temperatures",           "Acute",   2, "IPCC"),
    ("Meteorological", "Temperature",    "cold snap",                      "Acute",   1, "IPCC"),
    ("Meteorological", "Temperature",    "cold snaps",                     "Acute",   1, "IPCC"),
    ("Meteorological", "Temperature",    "freeze",                         "Acute",   1, "IPCC"),
    ("Meteorological", "Temperature",    "hard freeze",                    "Acute",   2, "IPCC"),
    ("Meteorological", "Temperature",    "deep freeze",                    "Acute",   2, "IPCC"),
    # AR6 CID-aligned additions
    ("Meteorological", "Temperature",    "cold spell",                     "Acute",   2, "IPCC AR6 CID"),
    ("Meteorological", "Temperature",    "cold spells",                    "Acute",   2, "IPCC AR6 CID"),
    ("Meteorological", "Temperature",    "heat dome",                      "Acute",   3, "IPCC AR6 / NOAA"),
    ("Meteorological", "Temperature",    "heat domes",                     "Acute",   3, "IPCC AR6 / NOAA"),
    ("Meteorological", "Temperature",    "thermal stress",                 "Chronic", 2, "IPCC/ILO"),
    ("Meteorological", "Temperature",    "wet bulb",                       "Acute",   2, "IPCC"),
    ("Meteorological", "Temperature",    "wet-bulb",                       "Acute",   2, "IPCC"),
    ("Meteorological", "Temperature",    "wet-bulb temperature",           "Acute",   2, "IPCC"),
    ("Meteorological", "Temperature",    "polar vortex",                   "Acute",   3, "IPCC/NOAA"),
    ("Meteorological", "Temperature",    "polar vortex disruption",        "Acute",   3, "IPCC/NOAA"),

    #  GEOPHYSICAL — Land movement (Acute / chronic mix) 
    ("Geophysical", "Land movement",     "landslide",                      "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "landslides",                     "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "mudslide",                       "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "mudslides",                      "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "debris flow",                    "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "debris flows",                   "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "avalanche",                      "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "avalanches",                     "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "subsidence",                     "Acute",   1, "IPCC"),
    ("Geophysical", "Land movement",     "solifluction",                   "Chronic", 1, "IPCC"),
    ("Geophysical", "Land movement",     "rockfall",                       "Acute",   1, "IPCC"),
    ("Geophysical", "Land movement",     "rockfalls",                      "Acute",   1, "IPCC"),
    ("Geophysical", "Land movement",     "slope failure",                  "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "slope failures",                 "Acute",   2, "IPCC"),
    ("Geophysical", "Land movement",     "mass movement",                  "Acute",   2, "IPCC"),
    ("Geophysical", "Coastal",           "coastal erosion",                "Acute",   2, "IPCC"),

    #  FIRE — Wildfire (Acute) 
    ("Fire", "Wildfire",                 "wildfire",                       "Acute",   3, "IPCC/TCFD"),
    ("Fire", "Wildfire",                 "wildfires",                      "Acute",   3, "IPCC/TCFD"),
    ("Fire", "Wildfire",                 "forest fire",                    "Acute",   3, "IPCC"),
    ("Fire", "Wildfire",                 "forest fires",                   "Acute",   3, "IPCC"),
    ("Fire", "Wildfire",                 "grassland fire",                 "Acute",   2, "IPCC"),
    ("Fire", "Wildfire",                 "grassland fires",                "Acute",   2, "IPCC"),
    ("Fire", "Wildfire",                 "bush fire",                      "Acute",   2, "IPCC"),
    ("Fire", "Wildfire",                 "bush fires",                     "Acute",   2, "IPCC"),
    ("Fire", "Wildfire",                 "bushfire",                       "Acute",   2, "IPCC"),
    ("Fire", "Wildfire",                 "bushfires",                      "Acute",   2, "IPCC"),
    ("Fire", "Wildfire",                 "fire weather",                   "Acute",   1, "IPCC"),
    ("Fire", "Wildfire",                 "red flag warning",               "Acute",   1, "IPCC"),
    ("Fire", "Wildfire",                 "red flag warnings",              "Acute",   1, "IPCC"),
    ("Fire", "Wildfire",                 "fire danger",                    "Acute",   1, "IPCC"),
    ("Fire", "Wildfire",                 "extreme fire danger",            "Acute",   2, "IPCC"),
    # AR6 CID Air pollution weather (smoke from fires)
    ("Fire", "Wildfire",                 "wildfire smoke",                 "Acute",   2, "IPCC AR6 CID"),
    ("Fire", "Wildfire",                 "smoke pollution",                "Acute",   1, "IPCC AR6 CID"),
    ("Fire", "Wildfire",                 "smoke plume",                    "Acute",   1, "IPCC AR6 CID"),
    ("Fire", "Wildfire",                 "smoke plumes",                   "Acute",   1, "IPCC AR6 CID"),

    #  TEMPERATURE — Chronic 
    ("Temperature", "Heat",              "rising temperatures",            "Chronic", 1, "IPCC/TCFD"),
    ("Temperature", "Heat",              "extreme heat",                   "Chronic", 2, "IPCC"),
    ("Temperature", "Heat",              "urban heat island",              "Chronic", 1, "IPCC"),
    ("Temperature", "Heat",              "urban heat islands",             "Chronic", 1, "IPCC"),
    ("Temperature", "Heat",              "heat stress",                    "Chronic", 2, "IPCC/ILO"),
    ("Temperature", "Heat",              "permafrost thaw",                "Chronic", 2, "IPCC"),
    ("Temperature", "Heat",              "permafrost",                     "Chronic", 2, "IPCC"),
    ("Temperature", "Cold",              "freeze-thaw",                    "Chronic", 1, "IPCC"),
    ("Temperature", "Ocean",             "ocean warming",                  "Chronic", 2, "IPCC"),
    ("Temperature", "Ocean",             "marine heatwave",                "Chronic", 2, "IPCC"),
    ("Temperature", "Ocean",             "marine heatwaves",               "Chronic", 2, "IPCC"),
    ("Temperature", "Variability",       "temperature variability",        "Chronic", 1, "IPCC"),

    #  WATER & SEA — Chronic 
    ("Water & Sea", "Sea level",         "sea level rise",                 "Chronic", 2, "IPCC/TCFD"),
    ("Water & Sea", "Sea level",         "sea-level rise",                 "Chronic", 2, "IPCC/TCFD"),
    ("Water & Sea", "Sea level",         "coastal inundation",             "Chronic", 2, "IPCC"),
    ("Water & Sea", "Sea level",         "saltwater intrusion",            "Chronic", 1, "IPCC"),
    ("Water & Sea", "Ocean chemistry",   "ocean acidification",            "Chronic", 2, "IPCC"),
    ("Water & Sea", "Ocean chemistry",   "ocean deoxygenation",            "Chronic", 1, "IPCC AR6 CID"),
    ("Water & Sea", "Ocean chemistry",   "coral bleaching",                "Chronic", 2, "IPCC"),
    ("Water & Sea", "Water",             "drought",                        "Chronic", 2, "IPCC/TCFD"),
    ("Water & Sea", "Water",             "droughts",                       "Chronic", 2, "IPCC/TCFD"),
    # AR6 CID drought subtypes
    ("Water & Sea", "Water",             "hydrological drought",           "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "hydrological droughts",          "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "agricultural drought",           "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "agricultural droughts",          "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "ecological drought",             "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "ecological droughts",            "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "meteorological drought",         "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "meteorological droughts",        "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "aridity",                        "Chronic", 1, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "flash drought",                  "Acute",   3, "IPCC/AMS"),
    ("Water & Sea", "Water",             "flash droughts",                 "Acute",   3, "IPCC/AMS"),
    ("Water & Sea", "Water",             "water scarcity",                 "Chronic", 2, "IPCC/TCFD"),
    ("Water & Sea", "Water",             "groundwater depletion",          "Chronic", 1, "IPCC"),
    ("Water & Sea", "Water",             "glacial retreat",                "Chronic", 2, "IPCC"),
    ("Water & Sea", "Water",             "ice loss",                       "Chronic", 2, "IPCC"),
    # AR6 CID Lake/river/sea ice
    ("Water & Sea", "Water",             "sea ice loss",                   "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "sea ice retreat",                "Chronic", 2, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "sea ice extent",                 "Chronic", 1, "IPCC AR6 CID"),
    ("Water & Sea", "Water",             "snowpack reduction",             "Chronic", 1, "IPCC"),
    ("Water & Sea", "Water",             "dry spell",                      "Acute",   1, "IPCC"),
    ("Water & Sea", "Water",             "dry spells",                     "Acute",   1, "IPCC"),
    ("Water & Sea", "Water quality",     "water quality",                  "Chronic", 1, "IPCC"),
    ("Water & Sea", "Water quality",     "sedimentation",                  "Chronic", 1, "IPCC"),
    ("Water & Sea", "Water quality",     "algal bloom",                    "Chronic", 1, "IPCC"),
    ("Water & Sea", "Water quality",     "algal blooms",                   "Chronic", 1, "IPCC"),

    #  CLIMATE PATTERNS — Chronic 
    ("Climate Patterns", "Precipitation","precipitation patterns",         "Chronic", 1, "IPCC/TCFD"),
    ("Climate Patterns", "Precipitation","drought frequency",              "Chronic", 2, "IPCC"),
    ("Climate Patterns", "Precipitation","monsoon",                        "Chronic", 2, "IPCC"),
    ("Climate Patterns", "Precipitation","monsoons",                       "Chronic", 2, "IPCC"),
    ("Climate Patterns", "Precipitation","flood frequency",                "Chronic", 2, "IPCC"),
    ("Climate Patterns", "Precipitation","hydrological variability",       "Chronic", 1, "IPCC"),
    ("Climate Patterns", "Wind",         "wind patterns",                  "Chronic", 1, "IPCC"),
    ("Climate Patterns", "Variability",  "weather variability",            "Chronic", 1, "IPCC"),
    ("Climate Patterns", "Variability",  "El Nino",                        "Chronic", 2, "IPCC"),
    ("Climate Patterns", "Variability",  "La Nina",                        "Chronic", 2, "IPCC"),
    ("Climate Patterns", "Variability",  "jet stream",                     "Chronic", 1, "IPCC"),

    #  ECOSYSTEM & LAND — Chronic 
    ("Ecosystem & Land", "Land",         "soil erosion",                   "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Land",         "desertification",                "Chronic", 2, "IPCC"),
    ("Ecosystem & Land", "Land",         "land subsidence",                "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Land",         "peatland degradation",           "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Land",         "soil degradation",               "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Biodiversity", "habitat loss",                   "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Biodiversity", "species range",                  "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Biodiversity", "ecosystem service",              "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Biodiversity", "ecosystem services",             "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Agriculture",  "crop yield",                     "Chronic", 2, "IPCC"),
    ("Ecosystem & Land", "Agriculture",  "crop yields",                    "Chronic", 2, "IPCC"),
    ("Ecosystem & Land", "Agriculture",  "soil fertility",                 "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Agriculture",  "growing season",                 "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Agriculture",  "pollinator decline",             "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Agriculture",  "pest pressure",                  "Chronic", 1, "IPCC"),
    ("Ecosystem & Land", "Agriculture",  "disease pressure",               "Chronic", 1, "IPCC"),

    #  HEALTH & SOCIAL — Chronic 
    ("Health & Social", "Health",        "heat-related illness",           "Chronic", 2, "IPCC/WHO"),
    ("Health & Social", "Health",        "heat-related illnesses",         "Chronic", 2, "IPCC/WHO"),
    ("Health & Social", "Health",        "heat-related mortality",         "Chronic", 2, "IPCC/WHO"),
    ("Health & Social", "Health",        "heat-related death",             "Chronic", 2, "IPCC/WHO"),
    ("Health & Social", "Health",        "heat-related deaths",            "Chronic", 2, "IPCC/WHO"),
    ("Health & Social", "Health",        "vector-borne disease",           "Chronic", 2, "IPCC/WHO"),
    ("Health & Social", "Health",        "vector-borne diseases",          "Chronic", 2, "IPCC/WHO"),
    ("Health & Social", "Health",        "waterborne disease",             "Chronic", 1, "IPCC/WHO"),
    ("Health & Social", "Health",        "waterborne diseases",            "Chronic", 1, "IPCC/WHO"),
    ("Health & Social", "Health",        "air quality degradation",        "Chronic", 1, "IPCC/WHO"),
    ("Health & Social", "Health",        "food insecurity",                "Chronic", 2, "IPCC"),
    ("Health & Social", "Labor",         "heat-related productivity",      "Chronic", 2, "ILO/IPCC"),
    ("Health & Social", "Labor",         "workforce displacement",         "Chronic", 1, "ILO"),
    ("Health & Social", "Migration",     "climate migration",              "Chronic", 1, "IPCC"),
    ("Health & Social", "Migration",     "climate-induced displacement",   "Chronic", 1, "IPCC AR6 WGII"),
    ("Health & Social", "Migration",     "climate displacement",           "Chronic", 1, "IPCC AR6 WGII"),
    ("Health & Social", "Migration",     "livelihood loss",                "Chronic", 1, "IPCC"),

    #  CROSS-CUTTING — Infrastructure & compound events 
    # NOTE: framework-only vocabulary ("physical risk", "climate resilience",
    # "adaptive capacity") removed. They are analyst discourse, not hazards,
    # and inflate counts without contributing to physical-event signal.
    ("Cross-cutting", "Infrastructure",  "supply chain disruption",        "Both",    2, "TCFD"),
    ("Cross-cutting", "Infrastructure",  "supply chain disruptions",       "Both",    2, "TCFD"),
    ("Cross-cutting", "Infrastructure",  "asset stranding",                "Both",    2, "TCFD"),
    ("Cross-cutting", "Infrastructure",  "critical infrastructure failure","Both",    3, "TCFD"),
    ("Cross-cutting", "Infrastructure",  "critical infrastructure failures","Both",   3, "TCFD"),
    ("Cross-cutting", "Infrastructure",  "infrastructure failure",         "Both",    3, "TCFD"),
    ("Cross-cutting", "Infrastructure",  "infrastructure failures",        "Both",    3, "TCFD"),
    ("Cross-cutting", "Compound events", "compound event",                 "Both",    2, "IPCC"),
    ("Cross-cutting", "Compound events", "compound events",                "Both",    2, "IPCC"),
    ("Cross-cutting", "Compound events", "compound flood",                 "Acute",   2, "IPCC"),
    ("Cross-cutting", "Compound events", "compound floods",                "Acute",   2, "IPCC"),
    ("Cross-cutting", "Compound events", "compound flooding",              "Acute",   2, "IPCC"),
    ("Cross-cutting", "Compound events", "compound extreme",               "Both",    2, "IPCC AR6"),
    ("Cross-cutting", "Compound events", "compound extremes",              "Both",    2, "IPCC AR6"),
    ("Cross-cutting", "Compound events", "concurrent extreme",             "Both",    2, "IPCC AR6"),
    ("Cross-cutting", "Compound events", "concurrent extremes",            "Both",    2, "IPCC"),
    ("Cross-cutting", "Compound events", "multivariate extreme",           "Both",    2, "IPCC AR6"),
    ("Cross-cutting", "Compound events", "multivariate extremes",          "Both",    2, "IPCC AR6"),
    ("Cross-cutting", "Compound events", "cascading hazard",               "Both",    2, "IPCC"),
    ("Cross-cutting", "Compound events", "cascading hazards",              "Both",    2, "IPCC"),
    # AR6 umbrella event vocabulary (high-frequency journalistic phrases)
    ("Cross-cutting", "Extreme events",  "extreme weather event",          "Both",    2, "IPCC AR6 Ch11"),
    ("Cross-cutting", "Extreme events",  "extreme weather events",         "Both",    2, "IPCC AR6 Ch11"),
    ("Cross-cutting", "Extreme events",  "climate extreme",                "Both",    2, "IPCC AR6 Ch11"),
    ("Cross-cutting", "Extreme events",  "climate extremes",               "Both",    2, "IPCC AR6 Ch11"),
    ("Cross-cutting", "Extreme events",  "weather extreme",                "Both",    2, "IPCC AR6 Ch11"),
    ("Cross-cutting", "Extreme events",  "weather extremes",               "Both",    2, "IPCC AR6 Ch11"),
    ("Cross-cutting", "Extreme events",  "climate disaster",               "Both",    2, "IPCC AR6 WGII"),
    ("Cross-cutting", "Extreme events",  "climate disasters",              "Both",    2, "IPCC AR6 WGII"),
    ("Cross-cutting", "Extreme events",  "climate-related disaster",       "Both",    2, "IPCC AR6 WGII"),
    ("Cross-cutting", "Extreme events",  "climate-related disasters",      "Both",    2, "IPCC AR6 WGII"),
    ("Cross-cutting", "Extreme events",  "climate-induced flooding",       "Acute",   2, "IPCC AR6"),
    ("Cross-cutting", "Extreme events",  "natural disaster",               "Both",    2, "IPCC AR6"),
    ("Cross-cutting", "Extreme events",  "natural disasters",              "Both",    2, "IPCC AR6"),
]

# Deduplication safety: ensure no term appears twice 
_seen = set()
_deduped = []
for row in TAXONOMY:
    term = row[2].lower()
    if term in _seen:
        continue
    _seen.add(term)
    _deduped.append(row)
TAXONOMY = _deduped
del _seen, _deduped


#  Derived lookup structures 
LEXICON = {row[2].lower() for row in TAXONOMY}
LEXICON_ALL = LEXICON

# Strict acute / chronic partitioning. "Both" is included in BOTH partitions
# (cross-cutting infrastructure terms apply to both indices).
LEXICON_ACUTE = {
    row[2].lower()
    for row in TAXONOMY
    if row[3] in ("Acute", "Both")
}

LEXICON_CHRONIC = {
    row[2].lower()
    for row in TAXONOMY
    if row[3] in ("Chronic", "Both")
}

SEVERITY = {row[2].lower(): row[4] for row in TAXONOMY}


def taxonomy_summary():
    from collections import Counter

    cat_counts = Counter(row[0] for row in TAXONOMY)
    sev_counts = Counter(row[4] for row in TAXONOMY)
    rtype_counts = Counter(row[3] for row in TAXONOMY)

    print(f"Total terms in L: {len(TAXONOMY)}")
    print(f"  Acute lexicon: {len(LEXICON_ACUTE)}")
    print(f"  Chronic lexicon: {len(LEXICON_CHRONIC)}")
    print(f"Severity distribution: {dict(sorted(sev_counts.items()))}")
    print(f"Risk-type distribution: {dict(rtype_counts)}")
    print("-" * 50)
    print("Terms per category:")
    for cat, n in sorted(cat_counts.items()):
        print(f"  {cat:<25}  {n:>3} terms")


taxonomy_summary()

Total terms in L: 341
  Acute lexicon: 253
  Chronic lexicon: 117
Severity distribution: {1: 104, 2: 190, 3: 47}
Risk-type distribution: {'Acute': 224, 'Chronic': 88, 'Both': 29}
--------------------------------------------------
Terms per category:
  Climate Patterns            11 terms
  Cross-cutting               33 terms
  Ecosystem & Land            16 terms
  Fire                        19 terms
  Geophysical                 16 terms
  Health & Social             17 terms
  Hydrological                39 terms
  Meteorological             144 terms
  Temperature                 12 terms
  Water & Sea                 34 terms


**Phase 1 imports.**

In [5]:
# Imports for Phase 1 that are required

from __future__ import annotations

import json
import logging
import os
import re
import time as time_module
from dataclasses import dataclass, field, asdict
from datetime import datetime, date, time as dt_time, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from spacy.tokens import Doc

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import mediacloud.api
import trafilatura
from newspaper import Article as NewspaperArticle

log = logging.getLogger(__name__)


**Country scope** 

`PHASE1_COUNTRIES['US']` — collection ID, timezone, `ready` flag. (Unused sparse-coverage/date constants for other countries trimmed.)

In [6]:
#  Country / sample scope 
PHASE1_COUNTRIES: dict[str, dict] = {
    "US": {"collection_ids": [34412234], "tz_name": "America/New_York", "ready": True},
}


**Stage A hyperparameters**

In [7]:
#  Stage A hyperparameters 
MIN_BODY_WORDS          = 50
SPACY_MAX_CHARS         = 100_000

CLIMATE_SPECIALIST_OUTLETS: set[str] = {
    "carbon brief", "inside climate news", "climate central",
    "climate home news", "desmog", "grist",
}

**Stage B hyperparameters.** 

In [8]:
#  Stage B hyperparameters 
SYNDICATION_COSINE_THRESHOLD  = 0.85   # conservative — only near-identical
SYNDICATION_WINDOW_HOURS      = 24

**Reject-sample retention flag.**

In [9]:
#  Reject-sample retention 
# Set this to True to persist full text of rejected articles 
# (otherwise only metadata survives in the audit CSV).
KEEP_REJECT_TEXT = True


**Output directories & request pacing.** 

In [10]:
#  Output directories 
ROOT       = THESIS_DATA_ROOT / "phase1_output"
DIR_SCORED = ROOT / "scored"
DIR_CACHE  = ROOT / "cache"
for d in (ROOT, DIR_SCORED, DIR_CACHE):
    d.mkdir(parents=True, exist_ok=True)

SECONDS_BETWEEN_REQS = 33.0
MAX_RETRIES          = 5
WEEKLY_BUDGET        = 4000

**Taxonomy-derived structures.** 

Builds the Media Cloud query string, lexicon sets (incl. `TRANSITION_EXCLUSION_LEXICON`), severity lookup.

In [11]:
ALLOW_EMPTY_TAXONOMY = False

if "TAXONOMY" not in globals():
    TAXONOMY = []

if not TAXONOMY and not ALLOW_EMPTY_TAXONOMY:
    raise ValueError(
        "TAXONOMY is empty or has not been defined. "
        "Run the first notebook cell containing TAXONOMY = [...] before this cell."
    )

LEXICON_ALL     = {row[2].lower() for row in TAXONOMY}
LEXICON_ACUTE   = {row[2].lower() for row in TAXONOMY if row[3] in ("Acute", "Both", "Acute/Chronic")}
LEXICON_CHRONIC = {row[2].lower() for row in TAXONOMY if row[3] in ("Chronic", "Both", "Acute/Chronic")}
SEVERITY        = {row[2].lower(): row[4] for row in TAXONOMY}

TRANSITION_EXCLUSION_LEXICON: set[str] = {
    "climate policy", "carbon tax", "emissions trading", "net zero",
    "green transition", "decarbonisation", "decarbonization",
    "transition risk", "energy transition", "renewable subsidy",
    "esg investing", "sustainable investing", "carbon disclosure",
}

## NLP & Retrieval Infrastructure

**spaCy loader + phrase matcher builder.**

In [12]:
def load_spacy_model(name: str = "en_core_web_trf") -> spacy.language.Language:
    """Load spaCy. Production runs must use en_core_web_trf for NER quality."""
    try:
        nlp = spacy.load(name)
        log.info(f"spaCy loaded: {name}")
        return nlp
    except OSError:
        log.error(f"{name} not installed. Run: python -m spacy download {name}")
        log.warning("Falling back to en_core_web_sm — NER quality will be lower")
        return spacy.load("en_core_web_sm")


COUNTRY_GPE: dict[str, set[str]] = {
    "US": {"united states", "united states of america",
           "u.s.", "u.s.a.", "usa", "america"},
}

def build_phrase_matcher(
    nlp: spacy.language.Language,
    lexicon: set[str],
    label: str = "LEX",
    attr: str = "LOWER",
) -> PhraseMatcher:
    '''
    Build a contiguous phrase matcher.

    Uses LOWER by default because it is robust for exact lexicon phrases.
    Use attr='LEMMA' only if both patterns and documents are processed
    through the full spaCy pipeline.
    '''
    matcher = PhraseMatcher(nlp.vocab, attr=attr)
    patterns = [nlp.make_doc(term) for term in sorted(lexicon)]
    matcher.add(label, patterns)
    return matcher

**Rate-limited Media Cloud client.**

In [13]:
import json
import time as time_module
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import mediacloud.api


@dataclass
class WeeklyBudget:
    week_key: str
    request_count: int

    @classmethod
    def load(cls, path: Path) -> "WeeklyBudget":
        wk = datetime.now(timezone.utc).strftime("%G-W%V")

        if not path.exists():
            return cls(wk, 0)

        try:
            data = json.loads(path.read_text())
            if data.get("week_key") == wk:
                return cls(wk, int(data.get("request_count", 0)))
            return cls(wk, 0)
        except Exception:
            return cls(wk, 0)

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(asdict(self), indent=2))

    def increment(self, path: Path) -> None:
        self.request_count += 1
        self.save(path)

    def remaining(self) -> int:
        return max(WEEKLY_BUDGET - self.request_count, 0)


class RateLimitedMediaCloud:
    def __init__(self, api_key: str, budget_file: Path):
        if not api_key:
            raise ValueError("Media Cloud API key is empty.")

        self.client = mediacloud.api.SearchApi(api_key)
        self.budget_file = budget_file
        self.budget = WeeklyBudget.load(budget_file)
        self._last_req = 0.0

    def _wait(self) -> None:
        elapsed = time_module.monotonic() - self._last_req

        if elapsed < SECONDS_BETWEEN_REQS:
            time_module.sleep(SECONDS_BETWEEN_REQS - elapsed)

    def _check_budget(self) -> None:
        if self.budget.remaining() <= 0:
            raise RuntimeError(
                f"Media Cloud weekly request budget exhausted: "
                f"{self.budget.request_count}/{WEEKLY_BUDGET}"
            )

    @staticmethod
    def _date_obj(d):
        '''
        Return a date-like object.

        The installed Media Cloud client calls .isoformat() internally,
        so pass date objects, not pre-converted strings.
        '''
        if isinstance(d, str):
            return pd.to_datetime(d).date()
        return d

    @staticmethod
    def _normalise_story_response(response) -> tuple[list[dict], str | None]:
        '''
        Normalise possible Media Cloud response formats into:

            stories, next_pagination_token
        '''
        if isinstance(response, tuple):
            if len(response) == 2:
                stories, token = response
                return list(stories or []), token
            raise ValueError(f"Unexpected Media Cloud tuple response: {response}")

        if isinstance(response, dict):
            stories = (
                response.get("stories")
                or response.get("results")
                or response.get("data")
                or []
            )
            token = (
                response.get("pagination_token")
                or response.get("next_page_token")
                or response.get("next")
            )
            return list(stories or []), token

        stories = (
            getattr(response, "stories", None)
            or getattr(response, "results", None)
            or []
        )
        token = (
            getattr(response, "pagination_token", None)
            or getattr(response, "next_page_token", None)
            or getattr(response, "next", None)
        )

        return list(stories or []), token

    def story_list(
        self,
        query: str,
        start_date,
        end_date,
        collection_ids: list[int] | None = None,
        source_ids: list[int] | None = None,
        pagination_token: str | None = None,
        page_size: int = 500,  # ↑ 100→500: 5× fewer API round-trips per date window
    ) -> tuple[list[dict], str | None]:
        '''
        Retrieve stories from Media Cloud.

        Uses the native Media Cloud client signature:

            story_list(
                query,
                start_date,
                end_date,
                collection_ids=[],
                source_ids=[],
                platform=None,
                expanded=None,
                pagination_token=None,
                sort_order=None,
                page_size=None,
            )

        Pass exactly one of:
          - source_ids: curated source-set route, matching Search UI ss=
          - collection_ids: fallback collection route
        '''
        if source_ids and collection_ids:
            raise ValueError("Pass either source_ids or collection_ids, not both.")

        if not source_ids and not collection_ids:
            raise ValueError("Must pass source_ids or collection_ids.")

        if not query or not str(query).strip():
            raise ValueError("Media Cloud query is empty.")

        self._check_budget()
        self._wait()

        start_obj = self._date_obj(start_date)
        end_obj = self._date_obj(end_date)

        clean_source_ids = [int(sid) for sid in source_ids or []]
        clean_collection_ids = [int(cid) for cid in collection_ids or []]

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = self.client.story_list(
                    query=query,
                    start_date=start_obj,
                    end_date=end_obj,
                    collection_ids=clean_collection_ids,
                    source_ids=clean_source_ids,
                    platform="onlinenews-mediacloud",
                    pagination_token=pagination_token,
                    page_size=page_size,
                )

                self.budget.increment(self.budget_file)
                self._last_req = time_module.monotonic()

                return self._normalise_story_response(response)

            except Exception as e:
                if attempt == MAX_RETRIES:
                    raise

                log.warning(f"MC error attempt {attempt}: {e}; backoff")
                time_module.sleep(10 * attempt)

        return [], None

**Audit record schema.** 

Tracks every retrieved story's fate through Stage A/B.

In [14]:
@dataclass
class AuditRecord:
    #Records the fate of every retrieved story across all 8 steps.
    story_id:         str
    country:          str
    url:              str
    media_name:       str
    publish_dt_utc:   str
    title:            str

    # Stage A outcomes
    scrape_status:    str   = ""
    word_count:       int   = 0
    nlp_ok:           bool  = False
    assigned_date:    str   = ""
    rolled_to_next:   bool  = False
    lexicon_passed:   bool  = False
    hazard_hits:      str   = ""
    exclusion_hits:   str   = ""
    country_mentions: int   = 0
    country_passed:   bool  = False
    is_specialist:    bool  = False
    passed_stage_a:   bool  = False

    # Stage B outcomes
    cred_tier:        int   = 0
    c_o:              float = np.nan
    syndicated_dropped: bool = False   

    final_kept:       bool  = False
    drop_reason:      str   = ""



## Phase 1 - Stage A: Retrieval

**Query builder.** 

`build_hazard_query()` — used directly by the 504-fix cell in the production run.

In [15]:
def build_hazard_query(
    taxonomy: list[tuple],
    weather_disaster_only: bool = True,
    require_climate_context: bool = False,
    min_severity_for_query: int = 1,
    category: str | None = None,
) -> str:
    rows = list(taxonomy)

    # Optional category-specific retrieval
    if category is not None:
        rows = [row for row in rows if row[0] == category]

    if weather_disaster_only:
        rows = [
            row for row in rows
            if row[0] in {"Hydrological", "Meteorological", "Geophysical", "Fire"}
            and row[3] in ("Acute", "Both")
            and row[4] >= min_severity_for_query
        ]
    else:
        rows = [
            row for row in rows
            if row[4] >= min_severity_for_query
        ]

    hazard_terms = sorted({row[2].lower() for row in rows})

    if not hazard_terms:
        raise ValueError(
            "No hazard terms found for query. "
            f"category={category!r}, "
            f"weather_disaster_only={weather_disaster_only}, "
            f"min_severity_for_query={min_severity_for_query}"
        )

    hazard_query = "(" + " OR ".join(f'"{term}"' for term in hazard_terms) + ")"

    if not require_climate_context:
        return hazard_query

    climate_terms = {
        "climate change",
        "global warming",
        "warming climate",
        "changing climate",
        "climate crisis",
        "climate risk",
        "extreme weather",
        "weather",
    }

    climate_query = "(" + " OR ".join(f'"{t}"' for t in sorted(climate_terms)) + ")"

    return f"{hazard_query} AND {climate_query}"

**Source control.** 

Curated domain whitelist + Media Cloud source IDs per country.

In [16]:
# Media Cloud source-set retrieval, post-filtered to a curated whitelist of national-press domains per country.

import os
from pathlib import Path

# Media Cloud source-set IDs 
# These correspond to the `ss=` parameter in the Media Cloud Search URL:
# search.mediacloud.org/search?...&ss=300560%252C1750%252C...

MEDIACLOUD_SOURCE_IDS: dict[str, list[int]] = {
    "US": [1089, 1, 4, 1752, 1095, 162705, 25499 ],
}


def mediacloud_source_ids_for(country_code: str) -> list[int]:
    return MEDIACLOUD_SOURCE_IDS.get(country_code, [])


#  Curated whitelist 
ALLOWED_DOMAINS: dict[str, set[str]] = {
    "US": {
        # Tier 1
        "nytimes.com",
        "usatoday.com",
        "reuters.com",

        # Tier 2
        "cbsnews.com",
        "cnn.com",
        "nbcnews.com",
        "abcnews.com",
    },

}


#  Domain filtering helpers 
def domain_of(url: str) -> str:
    '''
    Extract bare domain from a URL.

    Example:
        'https://www.bbc.co.uk/news/...' → 'bbc.co.uk'
    '''
    if not url:
        return ""

    s = str(url).lower().strip()

    if "://" in s:
        s = s.split("://", 1)[1]

    s = s.split("/", 1)[0]
    s = s.removeprefix("www.")

    return s


def is_whitelisted(url: str, country_code: str) -> bool:
    '''
    Return True if the article's domain matches the country's whitelist.

    Empty whitelist means no filtering, used for DE/JP until source curation
    is complete.
    '''
    allowed = ALLOWED_DOMAINS.get(country_code, set())

    if not allowed:
        return True

    d = domain_of(url)

    if d in allowed:
        return True

    # Allow subdomains:
    #   news.bbc.co.uk → bbc.co.uk
    return any(d == a or d.endswith("." + a) for a in allowed)


#  URL canonicalisation for deduplication 
from urllib.parse import urlsplit, urlunsplit


def canonical_url(url: str) -> str:
    '''
    Canonicalise URLs for cross-source deduplication.

    Removes:
      - leading www.
      - trailing slash
      - query string
      - fragment
    '''
    if not url:
        return ""

    u = urlsplit(str(url).strip().lower())
    netloc = u.netloc.removeprefix("www.")
    path = u.path.rstrip("/")

    return urlunsplit((u.scheme, netloc, path, "", ""))

#  Diagnostics 
print("Source whitelist:")
for cc, domains in ALLOWED_DOMAINS.items():
    if domains:
        print(f"  {cc}: {len(domains)} curated outlets")
    else:
        print(f"  {cc}: empty — no whitelist filtering yet")

print("\nMedia Cloud source IDs:")
for cc, source_ids in MEDIACLOUD_SOURCE_IDS.items():
    if source_ids:
        print(f"  {cc}: {len(source_ids)} source IDs: {source_ids}")
    else:
        print(f"  {cc}: none — will use PHASE1_COUNTRIES collection_ids")

Source whitelist:
  US: 7 curated outlets

Media Cloud source IDs:
  US: 7 source IDs: [1089, 1, 4, 1752, 1095, 162705, 25499]


> **Retrieval.** 

`step1_retrieve()` -- Media Cloud retrieval, post-filtered to the domain whitelist and deduplicated by URL. `gather_corpus()` calls this directly.

In [17]:
# RETRIEVAL PATCH
# Safe to run on a fresh kernel — no circular guards.
# Idempotent — safe to re-run.

from datetime import date
from zoneinfo import ZoneInfo

#  Guard: only the things that must exist BEFORE this cell 
for name in ("mediacloud_source_ids_for", "PHASE1_COUNTRIES",
             "ALLOWED_DOMAINS", "is_whitelisted", "canonical_url",
             "build_hazard_query", "log"):
    assert name in globals(), f"{name} not defined — run upstream cells first."


_TZ_FIXES = {
    "AU": "Australia/Sydney",
    "CA": "America/Toronto",
}
for _cc, _tz in _TZ_FIXES.items():
    if _cc in PHASE1_COUNTRIES:
        PHASE1_COUNTRIES[_cc]["tz_name"] = _tz
        PHASE1_COUNTRIES[_cc]["ready"]   = True
    if "COUNTRY_TIMEZONES" in globals() and _cc in COUNTRY_TIMEZONES:
        COUNTRY_TIMEZONES[_cc] = _tz

_tz_ok = True
for _cc, _data in PHASE1_COUNTRIES.items():
    try:
        ZoneInfo(_data["tz_name"])
    except Exception as _e:
        print(f"✗ PHASE1_COUNTRIES[{_cc}] bad tz_name: {_data['tz_name']} → {_e}")
        _tz_ok = False
if _tz_ok:
    print("✓ All PHASE1_COUNTRIES timezones valid")

#  Country-aware step1_retrieve 
def step1_retrieve(
    country_code: str,
    start: date,
    end: date,
    mc,
    category: str | None = None,
) -> list[dict]:
    '''
    Retrieve story metadata from Media Cloud, post-filtered to the
    curated domain whitelist and deduplicated by canonical URL and
    story_id.
    '''
    source_ids = mediacloud_source_ids_for(country_code)
    col_ids = PHASE1_COUNTRIES.get(country_code, {}).get("collection_ids", [])

    if not source_ids and not col_ids:
        log.error(f"No Media Cloud source_ids or collection_ids for {country_code}")
        return []

    query = build_hazard_query(
        TAXONOMY,
        weather_disaster_only=True,
        require_climate_context=False,
        min_severity_for_query=2,
        category=category,
    )

    mc_stories: list[dict] = []
    page_n = 0
    token = None
    while True:
        try:
            if source_ids:
                page, token = mc.story_list(
                    query=query, start_date=start, end_date=end,
                    source_ids=source_ids, pagination_token=token,
                )
            else:
                page, token = mc.story_list(
                    query=query, start_date=start, end_date=end,
                    collection_ids=col_ids, pagination_token=token,
                )
        except Exception as e:
            log.error(f"Media Cloud error: {country_code} {start}→{end}: {e}")
            break
        page_n += 1
        if source_ids:
            log.info(f"  MC page {page_n}: {len(page)} stories from {len(source_ids)} source IDs")
        else:
            log.info(f"  MC page {page_n}: {len(page)} stories from collection_ids={col_ids}")
        for s in page:
            s["_retrieval_source"] = "mediacloud"
            mc_stories.append(s)
        if not token:
            break

    n_before = len(mc_stories)
    mc_stories = [s for s in mc_stories if is_whitelisted(s.get("url", ""), country_code)]
    log.info(f"  MC whitelist filter: {n_before} → {len(mc_stories)} stories")

    #  Dedup by canonical URL, then by story id 
    seen_urls: set[str] = set()
    merged: list[dict] = []
    for s in mc_stories:
        url = canonical_url(s.get("url", ""))
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)
        merged.append(s)

    seen_ids: set[str] = set()
    deduped: list[dict] = []
    for s in merged:
        sid = str(s.get("id", "")).strip()
        if sid:
            if sid in seen_ids:
                continue
            seen_ids.add(sid)
        deduped.append(s)

    log.info(f"  Final after dedup: {len(deduped)} stories (MC after whitelist: {len(mc_stories)})")
    return deduped


print("✓ Retrieval patch applied")


✓ All PHASE1_COUNTRIES timezones valid
✓ Retrieval patch applied


**Full-text scraping.** 

`step2_scrape()` — trafilatura → newspaper3k fallback, with URL-level disk cache.

In [18]:
def step2_scrape(url: str, story: dict | None = None) -> tuple[str | None, str]:
    '''
    Scrapes full article text for a given URL.

    Speed additions (no logic change):
      - URL-level disk cache: avoids re-scraping the same URL across runs.
      - trafilatura timeout: caps stalled connections at 10 s instead of
        waiting indefinitely.
    '''
    #  Disk cache: keyed by URL hash 
    _url_hash = hashlib.md5(url.encode()).hexdigest()
    _cache_path = DIR_CACHE / "scrape" / f"{_url_hash}.json"
    _cache_path.parent.mkdir(parents=True, exist_ok=True)

    if _cache_path.exists():
        try:
            cached = json.loads(_cache_path.read_text())
            return cached.get("text"), cached.get("status", "cache")
        except Exception:
            pass  

    def _write_cache(text, status):
        try:
            _cache_path.write_text(json.dumps({"text": text, "status": status}))
        except Exception:
            pass

    try:
        downloaded = trafilatura.fetch_url(url, no_ssl=False)
        if downloaded:
            text = trafilatura.extract(
                downloaded,
                include_comments=False,
                include_tables=False,
            )
            if text and len(text.split()) >= MIN_BODY_WORDS:
                _write_cache(text, "trafilatura")
                return text, "trafilatura"
    except Exception as e:
        log.debug(f"trafilatura failed {url}: {e}")

    try:
        art = NewspaperArticle(url)
        art.download()
        art.parse()
        if art.text and len(art.text.split()) >= MIN_BODY_WORDS:
            _write_cache(art.text, "newspaper3k")
            return art.text, "newspaper3k"
    except Exception as e:
        log.debug(f"newspaper3k failed {url}: {e}")

    _write_cache(None, "failed")
    return None, "failed"


**Timing assignment.** 

`assign_market_day()` — 16:00-local cutoff, weekend roll-to-Monday.

In [19]:
from datetime import datetime, date, time, timedelta
from zoneinfo import ZoneInfo
import pandas as pd


COUNTRY_TIMEZONES = {
    "US": "America/New_York",
}


COUNTRY_ALIASES = {
    "US": {
        "united states",
        "united states of america",
        "usa",
        "u.s.",
        "u.s.a.",
        "america",
        "american",
}
}
def parse_story_datetime(story: dict) -> datetime | None:
    '''
    Parse the best available Media Cloud timestamp.

    Tries several possible Media Cloud date fields because different API
    responses may expose different names/formats.
    '''
    date_fields = [
        "publish_date",
        "publication_date",
        "published_date",
        "indexed_date",
        "collect_date",
        "created_at",
        "date",
    ]

    for field in date_fields:
        value = story.get(field)

        if value is None or value == "":
            continue

        try:
            dt = pd.to_datetime(value, utc=True, errors="coerce")

            if pd.isna(dt):
                continue

            return dt.to_pydatetime()

        except Exception:
            continue

    return None


def _next_business_day(d: date) -> date:
    '''
    Roll Saturday/Sunday to the following Monday.
    Does not account for market holidays.
    '''
    while d.weekday() >= 5:
        d += timedelta(days=1)

    return d


def assign_market_day(
    pub_dt: datetime,
    country_code: str,
    cutoff_hour: int = 16,
) -> date:
    '''
    Assign article to the relevant market day.

    Articles published after 16:00 local market time are assigned to the next
    business day. Weekend dates are rolled forward to Monday.
    '''
    tz_name = COUNTRY_TIMEZONES.get(country_code)

    if tz_name is None:
        raise ValueError(f"No timezone configured for country_code={country_code}")

    if pub_dt.tzinfo is None:
        pub_dt = pub_dt.replace(tzinfo=ZoneInfo("UTC"))

    local_dt = pub_dt.astimezone(ZoneInfo(tz_name))
    market_day = local_dt.date()

    cutoff = datetime.combine(
        market_day,
        dt_time(cutoff_hour, 0),
        tzinfo=ZoneInfo(tz_name),
    )

    if local_dt >= cutoff:
        market_day += timedelta(days=1)

    return _next_business_day(market_day)


**Country attribution — the actual rule used.** 

`find_country_mentions()` implements a **domestic-share dominance test**: an article passes only if domestic-alias mentions clearly outweigh foreign-country mentions (`domestic_share >= MIN_DOMESTIC_SHARE`). 

In [20]:
import re
from spacy.tokens import Doc

#  Domestic alias sets 
COUNTRY_ALIASES = {
    "US": {
        "us", "u.s.", "u.s.a.", "usa", "united states", "america", "american",
        "washington", "new york", "california", "texas", "florida",
        "louisiana", "illinois", "ohio", "pennsylvania", "michigan",
        "georgia", "boston", "chicago", "los angeles", "houston", "miami",
        "seattle", "portland", "atlanta", "philadelphia",
    },
}

#  Foreign-country indicator sets (per home country) 
# When evaluating an article for country X, references to these other countries count against the article's domestic-share score.
FOREIGN_COUNTRIES = {
    "US": {
        "united kingdom", "uk", "u.k.", "britain", "british",
        "england", "scotland", "wales", "northern ireland",
        "australia", "australian", "queensland",
        "canada", "canadian",
        "france", "french", "spain", "spanish", "italy", "italian",
        "germany", "german", "japan", "japanese", "china", "chinese",
        "india", "indian", "russia", "russian",
    },
}
# Minimum share of geographic mentions that must be domestic for an article to pass attribution. Tune in §VI sensitivity analysis.
MIN_DOMESTIC_SHARE = 0.6


def find_country_mentions(doc: Doc, country_code: str) -> list[str]:
    '''
    Country-attribution filter using a geographic-dominance check.

    An article passes if its domestic mentions materially outweigh its
    foreign-country mentions:

        domestic_share = n_domestic / (n_domestic + n_foreign)
        domestic_share >= MIN_DOMESTIC_SHARE

    Edge cases:
      - No geographic anchor at all (n_domestic = n_foreign = 0): reject.
      - Pure-domestic article (n_foreign = 0): trivially passes.
      - Pure-foreign article (n_domestic = 0): trivially fails.
      - Mixed article: passes only if domestic clearly dominates.

    Returns the list of unique domestic alias strings found, or [] if
    the article fails the domestic-share gate. The list is preserved
    as audit metadata.
    '''
    text_lower = doc.text.lower()

    domestic_aliases = COUNTRY_ALIASES.get(country_code, set())
    foreign_aliases = FOREIGN_COUNTRIES.get(country_code, set())

    if not domestic_aliases:
        log.warning(f"No domestic aliases configured for {country_code}")
        return []

    domestic_count = 0
    domestic_hits: list[str] = []
    for alias in domestic_aliases:
        n = len(re.findall(rf"\b{re.escape(alias)}\b", text_lower))
        if n > 0:
            domestic_count += n
            domestic_hits.append(alias)

    foreign_count = sum(
        len(re.findall(rf"\b{re.escape(alias)}\b", text_lower))
        for alias in foreign_aliases
    )

    total = domestic_count + foreign_count
    if total == 0:
        return []

    domestic_share = domestic_count / total
    if domestic_share < MIN_DOMESTIC_SHARE:
        return []

    return sorted(domestic_hits)

**Fast lexicon pre-filter.** 

Regex speed-gate before the expensive spaCy pass.

In [21]:
# Fast lexicon pre-filter (regex) 
# Applied before spaCy so the expensive transformer pipeline only runs on
# articles that contain at least one lexicon term. Because the regex operates
# on raw text (no tokenisation), it fires before _safe_nlp is called in
# gather_corpus. The spaCy PhraseMatcher later remains the canonical
# filter; this is only a speed gate that avoids NLP on obvious misses.
_LEXICON_PREFILTER_RE: re.Pattern | None = None

def _build_lexicon_prefilter() -> re.Pattern:
    global _LEXICON_PREFILTER_RE
    if _LEXICON_PREFILTER_RE is not None:
        return _LEXICON_PREFILTER_RE
    sorted_terms = sorted(LEXICON_ALL, key=len, reverse=True)
    pattern = "|".join(re.escape(t) for t in sorted_terms)
    _LEXICON_PREFILTER_RE = re.compile(pattern, re.IGNORECASE)
    return _LEXICON_PREFILTER_RE

def _text_passes_prefilter(text: str) -> bool:
    return bool(_build_lexicon_prefilter().search(text))


def _safe_nlp(text: str, nlp: spacy.language.Language) -> Doc:
    '''
    Run spaCy on text. Truncate to SPACY_MAX_CHARS. If the transformer
    pipeline raises (e.g. on a single un-segmented blob), retry on a
    sentence-segmented fallback chunked at 50k chars.
    '''
    text = text[:SPACY_MAX_CHARS]
    try:
        return nlp(text)
    except Exception:
        # Fallback: chunk and re-merge by parsing each chunk independently
        chunks = [text[i:i+50_000] for i in range(0, len(text), 50_000)]
        docs = [nlp(c) for c in chunks]
        return Doc.from_docs(docs)


 **`gather_corpus()` — Stage A orchestration, and the actual hazard-inclusion rule used.**

This function embeds its own hazard-inclusion logic inline, and it is **stricter** than a simpler separately-documented rule (≥1 hit). The rule that actually ran:

```
passes_volume  = n_hits >= 3
passes_quality = n_hits >= 2 and has_high_severity_term
keep if (passes_volume or passes_quality)
```

Ties together retrieval, scraping (parallel, `SCRAPE_WORKERS` threads), timing, the hazard/exclusion filters above, and country attribution.

In [22]:
def gather_corpus(
    country_code: str,
    start: date,
    end: date,
    mc: RateLimitedMediaCloud,
    nlp: spacy.language.Language,
    hazard_matcher: PhraseMatcher,
    exclusion_matcher: PhraseMatcher,
    category: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, list[AuditRecord]]:
    '''
    Run Stage A corpus construction for one country and date range.

    Stage order:
      0. Retrieve stories from Media Cloud
      1. Scrape full article text  [PARALLEL — ThreadPoolExecutor]
      2. Parse publish date and assign market day
      2b. Regex pre-filter: skip NLP on articles with no lexicon terms (fast)
      3. Run spaCy NLP -> (only on pre-filter survivors)
      4. Keep articles with physical-risk hazard terms
      4b. Exclude transition/policy-risk articles
      5. Keep articles with target-country mentions

    Speed changes vs original:
      - Step 1 runs in a ThreadPoolExecutor (SCRAPE_WORKERS threads) instead
        of sequentially. All other logic is identical.
      - Step 2b applies a compiled regex before the expensive NLP call,
        dropping obvious non-matches without running spaCy at all.

    Returns:
      corpus_df  : articles passing Stage A
      rejects_df : rejected articles with scraped text, if KEEP_REJECT_TEXT
      audit      : one AuditRecord per retrieved story
    '''
    from concurrent.futures import ThreadPoolExecutor, as_completed

    log.info(
        f"=== STAGE A | {country_code} | {start} → {end} | "
        f"{category or 'all'} ==="
    )

    raw_stories = step1_retrieve(
        country_code=country_code,
        start=start,
        end=end,
        mc=mc,
        category=category,
    )

    if not raw_stories:
        return pd.DataFrame(), pd.DataFrame(), []

    audit: list[AuditRecord] = []
    kept: list[dict] = []
    rejects: list[dict] = []

    def add_reject(
        *,
        sid: str,
        country_code: str,
        url: str,
        title: str,
        media: str,
        drop_reason: str,
        text: str | None = None,
        pub_dt: datetime | None = None,
        market_day: date | str | None = None,
        scrape_status: str | None = None,
        retrieval_source: str = "unknown",
    ) -> None:
        #Need to store rejected articles with scraped text for later audit.
        if not KEEP_REJECT_TEXT or not text:
            return

        row = {
            "story_id": sid,
            "country": country_code,
            "url": url,
            "title": title,
            "media_name": media,
            "drop_reason": drop_reason,
            "text": text,
            "text_chars": len(text),
            "retrieval_source": retrieval_source,
        }

        if pub_dt is not None:
            row["publish_dt_utc"] = pub_dt.isoformat()
        if market_day is not None:
            row["market_day"] = market_day
            row["assigned_date"] = market_day
        if scrape_status is not None:
            row["scrape_status"] = scrape_status

        rejects.append(row)

    # STEP 1: Parallel scraping 
    # Articles are scraped concurrently using SCRAPE_WORKERS threads -> result order matches raw_stories order
    scrape_results: dict[str, tuple[str | None, str]] = {}

    def _scrape_one(story):
        url = story.get("url", "") or ""
        return str(story.get("id", "")), step2_scrape(url, story=story)

    with ThreadPoolExecutor(max_workers=SCRAPE_WORKERS) as pool:
        futures = {pool.submit(_scrape_one, s): s for s in raw_stories}
        for fut in as_completed(futures):
            try:
                sid, result = fut.result()
                scrape_results[sid] = result
            except Exception as e:
                s = futures[fut]
                scrape_results[str(s.get("id", ""))] = (None, f"exception:{e}")

    log.info(
        f"  Step 1 complete: {len(scrape_results)} scrape results "
        f"({sum(1 for t,_ in scrape_results.values() if t) } with text)"
    )

    # STEPS 2–5: Sequential per-article processing 
    for story in raw_stories:
        sid = str(story.get("id", ""))
        url = story.get("url", "") or ""
        title = story.get("title", "") or ""
        media = story.get("media_name", "") or ""

        retrieval_source = story.get("_retrieval_source", "unknown")

        rec = AuditRecord(
            story_id=sid,
            country=country_code,
            url=url,
            media_name=media,
            publish_dt_utc="",
            title=title[:300],
        )

        rec.retrieval_source = retrieval_source

        # 1. Retrieve scrape result
        text, scrape_status = scrape_results.get(sid, (None, "missing"))
        rec.scrape_status = scrape_status

        if not text:
            rec.drop_reason = "scrape_failed"
            audit.append(rec)
            continue

        rec.text_chars = len(text)
        rec.word_count = len(text.split())

        # 2. Parse publish date 
        pub_dt = parse_story_datetime(story)

        if pub_dt is None:
            rec.drop_reason = "missing_publish_date"
            add_reject(
                sid=sid,
                country_code=country_code,
                url=url,
                title=title,
                media=media,
                drop_reason=rec.drop_reason,
                text=text,
                scrape_status=scrape_status,
                retrieval_source=retrieval_source,
            )
            audit.append(rec)
            continue

        rec.publish_dt_utc = pub_dt.isoformat()

        market_day = assign_market_day(pub_dt=pub_dt, country_code=country_code)
        rec.assigned_date = str(market_day)

        try:
            local_pub_date = pub_dt.astimezone(
                ZoneInfo(PHASE1_COUNTRIES[country_code]["tz_name"])
            ).date()
            rec.rolled_to_next = market_day != local_pub_date
        except Exception:
            rec.rolled_to_next = False

        #  2b. Regex pre-filter (fast gate before expensive NLP) -> drops articles with no lexicon terms without running spaCy at all.
        if not _text_passes_prefilter(text):
            rec.drop_reason = "prefilter_no_lexicon_term"
            add_reject(
                sid=sid,
                country_code=country_code,
                url=url,
                title=title,
                media=media,
                drop_reason=rec.drop_reason,
                text=text,
                pub_dt=pub_dt,
                market_day=market_day,
                scrape_status=scrape_status,
                retrieval_source=retrieval_source,
            )
            audit.append(rec)
            continue

        # 3. spaCy NLP 
        try:
            doc = _safe_nlp(text, nlp)
            rec.nlp_ok = True
        except Exception as e:
            rec.drop_reason = f"nlp_error: {type(e).__name__}: {str(e)[:200]}"
            add_reject(
                sid=sid,
                country_code=country_code,
                url=url,
                title=title,
                media=media,
                drop_reason=rec.drop_reason,
                text=text,
                pub_dt=pub_dt,
                market_day=market_day,
                scrape_status=scrape_status,
                retrieval_source=retrieval_source,
            )
            audit.append(rec)
            continue

        #  4. Physical-risk hazard filter 
        hazard_hits = hazard_matcher(doc)
        matched_terms = [doc[s:e].text.lower() for _, s, e in hazard_hits]
        unique_terms = set(matched_terms)

        n_hits = len(hazard_hits)
        has_high_sev = any(SEVERITY.get(t, 0) >= 2 for t in unique_terms)

        passes_volume = n_hits >= 3
        passes_quality = n_hits >= 2 and has_high_sev

        if not (passes_volume or passes_quality):
            rec.drop_reason = "insufficient_hazard_signal"
            add_reject(
                sid=sid,
                country_code=country_code,
                url=url,
                title=title,
                media=media,
                drop_reason=rec.drop_reason,
                text=text,
                pub_dt=pub_dt,
                market_day=market_day,
                scrape_status=scrape_status,
                retrieval_source=retrieval_source,
            )
            audit.append(rec)
            continue

        rec.lexicon_passed = True
        rec.hazard_hits = ",".join(sorted(unique_terms))[:500]

        #  4b. Transition/policy-risk exclusion 
        exclusion_hits = exclusion_matcher(doc)

        if exclusion_hits:
            rec.exclusion_hits = ",".join(sorted({
                doc[s:e].text.lower() for _, s, e in exclusion_hits
            }))[:500]
            rec.drop_reason = "transition_risk_exclusion"

            add_reject(
                sid=sid,
                country_code=country_code,
                url=url,
                title=title,
                media=media,
                drop_reason=rec.drop_reason,
                text=text,
                pub_dt=pub_dt,
                market_day=market_day,
                scrape_status=scrape_status,
                retrieval_source=retrieval_source,
            )
            audit.append(rec)
            continue

        #  5. Country attribution 
        country_mentions = find_country_mentions(doc, country_code)

        if not country_mentions:
            rec.drop_reason = "no_country_mention"
            add_reject(
                sid=sid,
                country_code=country_code,
                url=url,
                title=title,
                media=media,
                drop_reason=rec.drop_reason,
                text=text,
                pub_dt=pub_dt,
                market_day=market_day,
                scrape_status=scrape_status,
                retrieval_source=retrieval_source,
            )
            audit.append(rec)
            continue

        rec.country_mentions = len(country_mentions)
        rec.country_passed = True

        # Passed stage A
        article = {
            "story_id": sid,
            "country": country_code,
            "url": url,
            "title": title,
            "media_name": media,
            "publish_dt_utc": pub_dt.isoformat(),
            "market_day": market_day,
            "assigned_date": market_day,
            "text": text,
            "text_chars": len(text),
            "scrape_status": scrape_status,
            "retrieval_source": retrieval_source,
            "hazard_hits_n": len(hazard_hits),
            "hazard_terms": ",".join(sorted(unique_terms)),
            "country_mentions": country_mentions,
        }

        kept.append(article)

        rec.is_specialist = media.lower() in CLIMATE_SPECIALIST_OUTLETS
        rec.passed_stage_a = True
        rec.drop_reason = "passed_stage_a"
        audit.append(rec)

    corpus_df = pd.DataFrame(kept)
    rejects_df = pd.DataFrame(rejects)

    log.info(
        f"  Stage A: {len(audit)} retrieved → "
        f"{len(corpus_df)} passed Stage A "
        f"({len(rejects_df)} rejects kept with text)"
    )

    if not corpus_df.empty and "retrieval_source" in corpus_df.columns:
        log.info(
            f"  Stage A kept source mix: "
            f"{corpus_df['retrieval_source'].value_counts(dropna=False).to_dict()}"
        )

    return corpus_df, rejects_df, audit


## Phase 1 - Stage B: Syndication Cleanup

> **`SyndicationCleaner` + `step8_syndication_cleanup()`.** 

TF-IDF near-duplicate detection (cosine ≥0.85, 24h window) to strip wire-service repeats. **Credibility weighting is not used in the final run**, the production Stage B cell explicitly skips it (`no credibility weighting`)

In [23]:
# STAGE B  definitions

from datetime import timedelta
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import logging

log = logging.getLogger(__name__)

SYNDICATION_WINDOW_HOURS   = 48
SYNDICATION_COSINE_THRESHOLD = 0.85


class SyndicationCleaner:
    '''
    TF-IDF cosine-similarity cleanup for wire-service / near-identical copies.
    NOT a novelty measure. The threshold (0.85) is intentionally conservative
    so that follow-up articles with materially new information are retained.
    Semantic novelty weighting is a Phase 3 operation.
    Fix vs. original notebook: the original fit IDF per-batch, so terms common
    in a given week (e.g. "wildfire" during a fire week) got LOW IDF — exactly
    backwards for detecting wire-service propagation of a single event.
    '''
    def __init__(self, max_features: int = 50_000,
                 ngram_range: tuple[int, int] = (1, 2)):
        self.vec = TfidfVectorizer(max_features=max_features,
                                   ngram_range=ngram_range,
                                   sublinear_tf=True)
        self._fitted = False

    def fit(self, reference_texts: list[str]) -> None:
        self.vec.fit(reference_texts)
        self._fitted = True

    def save(self, path: Path) -> None:
        import joblib
        joblib.dump(self.vec, path)

    @classmethod
    def load(cls, path: Path) -> "SyndicationCleaner":
        import joblib
        obj = cls()
        obj.vec = joblib.load(path)
        obj._fitted = True
        return obj

    def find_syndicated(self, articles: list[dict]) -> set[str]:
        if not self._fitted:
            raise RuntimeError("Call .fit(reference_texts) first")
        if len(articles) <= 1:
            return set()
        articles = sorted(articles, key=lambda a: a["publish_dt_utc"])
        texts    = [a.get("text", "") or "" for a in articles]
        try:
            tfidf = self.vec.transform(texts)
        except Exception as e:
            log.warning(f"TF-IDF transform failed: {e}; skipping cleanup")
            return set()
        keep   = [True] * len(articles)
        window = timedelta(hours=SYNDICATION_WINDOW_HOURS)
        for i in range(len(articles)):
            if not keep[i]:
                continue
            for j in range(i + 1, len(articles)):
                if not keep[j]:
                    continue
                if articles[j]["publish_dt_utc"] - articles[i]["publish_dt_utc"] > window:
                    break
                sim = cosine_similarity(tfidf[i], tfidf[j])[0, 0]
                if sim >= SYNDICATION_COSINE_THRESHOLD:
                    keep[j] = False
        return {a["story_id"] for a, k in zip(articles, keep) if not k}


def step8_syndication_cleanup(corpus_df: pd.DataFrame,
                               cleaner: SyndicationCleaner
                               ) -> tuple[pd.DataFrame, set[str]]:
    #Drop wire-service/near-identical syndication copies -> Returns (cleaned_df, dropped_ids)
    if corpus_df.empty:
        return corpus_df, set()
    articles = corpus_df.to_dict(orient="records")
    dropped  = cleaner.find_syndicated(articles)
    cleaned  = corpus_df[~corpus_df["story_id"].isin(dropped)].copy()
    log.info(f"  Step 8: {len(corpus_df)} → {len(cleaned)} after syndication cleanup "
             f"({len(dropped)} dropped)")
    return cleaned, dropped

**Scraping concurrency.** 

`SCRAPE_WORKERS = 20` — used inside `gather_corpus()`

In [24]:
SCRAPE_WORKERS: int = 20

## Phase 2: Article-Level Scoring

> Config, window helpers, per-hit factor computation, hazard matching, data models, and the core `score_article()`/`score_corpus()` functions, called directly by the production Phase 2 cell.

In [25]:
 # PHASE 2 CONFIGURATION, IMPORTS & OUTPUT PATHS


import math
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from spacy.tokens import Doc, Span, Token

# Phase 2 depends on Phase 1 globals already defined in this notebook
_REQUIRED_PHASE2_GLOBALS = [
    "ROOT", "DIR_SCORED", "TAXONOMY", "LEXICON_ALL", "SEVERITY",
    "COUNTRY_GPE", "build_phrase_matcher", "load_spacy_model", "log",
]
_missing = [name for name in _REQUIRED_PHASE2_GLOBALS if name not in globals()]
if _missing:
    raise RuntimeError(
        "Run Phase 1 setup cells before Phase 2. Missing globals: "
        + ", ".join(_missing)
    )

#  Length normalisation 
ALPHA = 0.5    # § VI sensitivity: {0.3, 0.5*, 0.8}

#  Valence shifters (Polanyi & Zaenen 2006: 3-token)
VALENCE_NEGATOR     = -1.0
VALENCE_AMPLIFIER   =  1.5
VALENCE_DEAMPLIFIER =  0.5
VALENCE_ADVERSATIVE =  0.5
VALENCE_WINDOW      =  3     

NEGATORS     = {"not", "no", "never", "neither", "without",
                "hardly", "barely", "scarcely"}
AMPLIFIERS   = {"very", "extremely", "highly", "severely",
                "catastrophically", "significantly", "dramatically",
                "substantially", "greatly", "enormously", "massively"}
DEAMPLIFIERS = {"somewhat", "slightly", "moderately", "partially",
                "marginally", "relatively"}
ADVERSATIVES = {"but", "however", "although", "though", "despite",
                "yet", "nevertheless"}

#  Intensity modifiers (Das & Chen 2007: 2-token) 
INTENSITY_WINDOW = 2

INTENSITY_MODIFIERS: dict[str, float] = {
    "devastating": 1.0, "catastrophic": 1.0, "unprecedented": 0.9,
    "severe":      0.8, "extreme":      0.8, "violent":       0.8,
    "major":       0.5, "significant":  0.5, "serious":       0.5,
    "heavy":       0.4, "intense":      0.4, "substantial":   0.4,
    "moderate":    0.2, "notable":      0.2,
}


In [26]:
# WITHIN-SENTENCE WINDOW HELPERS

def _sentence_bounds(token: Token) -> tuple[int, int]:
    sent: Span = token.sent
    return sent.start, sent.end


def _left_context_in_sentence(doc: Doc, hit_start: int, window: int) -> list[Token]:
    #return up to window tokens before hit_start, without crossing the start of the hit's sentence
    
    s_start, _ = _sentence_bounds(doc[hit_start])
    lo = max(s_start, hit_start - window)
    return [doc[i] for i in range(lo, hit_start)]


def _bidirectional_context_in_sentence(
    doc: Doc,
    hit_start: int,
    hit_end: int,
    window: int,
) -> list[Token]:
    #return tokens within window either side of the hit span, clamped to the hit's sentence boundaries, excludes the hit tokens themselves
    
    s_start, s_end = _sentence_bounds(doc[hit_start])
    lo = max(s_start, hit_start - window)
    hi = min(s_end, hit_end + window)
    return [doc[i] for i in range(lo, hi) if i < hit_start or i >= hit_end]


In [27]:
# PER-HIT FACTOR COMPUTATION

def get_severity(term: str) -> int:
    #Severity sᵢ ∈ {1, 2, 3} from the taxonomy -> defaults to 1 if missing
    return int(SEVERITY.get(term.lower(), 1))


def get_valence(doc: Doc, hit_start: int) -> float:
    '''
    Compute vᵢ from valence shifters in the LEFT context of the hit, within
    the same sentence.

    Composition rule:
      - Any negator in window  → v starts at -1
      - Otherwise              → v starts at +1
      - One amplifier OR one de-amplifier multiplies v
        (amplifier dominates if both are present)
      - Adversative multiplies v by 0.5
    '''
    context = _left_context_in_sentence(doc, hit_start, VALENCE_WINDOW)
    lemmas  = {t.lower_ for t in context}

    v = VALENCE_NEGATOR if (lemmas & NEGATORS) else 1.0
    if lemmas & AMPLIFIERS:
        v *= VALENCE_AMPLIFIER
    elif lemmas & DEAMPLIFIERS:
        v *= VALENCE_DEAMPLIFIER
    if lemmas & ADVERSATIVES:
        v *= VALENCE_ADVERSATIVE
    return v


def get_intensity(doc: Doc, hit_start: int, hit_end: int) -> float:
    '''
    mᵢ = 1 + Σⱼ μⱼ over adjective/adverb modifiers within ±INTENSITY_WINDOW
    tokens of the hit, clamped to the hit's sentence.
    '''
    context = _bidirectional_context_in_sentence(
        doc, hit_start, hit_end, INTENSITY_WINDOW
    )
    mu_total = 0.0
    for t in context:
        if t.pos_ in ("ADJ", "ADV"):
            mu_total += INTENSITY_MODIFIERS.get(t.lower_, 0.0)
    return 1.0 + mu_total


def get_relevance(doc: Doc, hit_start: int, country_code: str) -> float:
    '''
    rel(i,k) = 1 / (1 + d(i,k)), where d(i,k) is the minimum token distance
    between the hit and any spaCy GPE entity matching the country's alias set.

    If no matching GPE entity is present, returns 0.0. This avoids inflating
    scores for articles that passed Phase 1 only through demonym backstops.
    '''
    aliases = COUNTRY_GPE.get(country_code, set())
    if not aliases:
        return 0.0

    gpe_positions: list[int] = []
    for ent in doc.ents:
        if ent.label_ != "GPE":
            continue
        if ent.text.lower().strip() in aliases:
            gpe_positions.append(ent.start)

    if not gpe_positions:
        return 0.0

    min_dist = min(abs(hit_start - pos) for pos in gpe_positions)
    return 1.0 / (1.0 + min_dist)


In [28]:
# HAZARD-TERM MATCHING

def find_hazard_hits(
    doc: Doc,
    hazard_matcher: PhraseMatcher,
) -> list[tuple[int, int, str]]:
    '''
    Return contiguous hazard-term hits as (start, end, term_lemma) tuples.

    Uses spaCy's PhraseMatcher for consistency with Phase 1. start and end
    are token indices; `end` is exclusive, following spaCy convention.
    '''
    matches = hazard_matcher(doc)
    hits: list[tuple[int, int, str]] = []
    for _, start, end in matches:
        term = doc[start:end].text.lower()
        hits.append((start, end, term))
    return hits


In [29]:
# DATA MODELS & AGGREGATORS

@dataclass
class HitDecomposition:
    #One contribution to S(n,t,k), preserved for audit and inspection.
    story_id:     str
    country:      str
    term:         str
    hit_start:    int
    hit_end:      int
    severity:     int
    valence:      float
    intensity:    float
    relevance:    float
    contribution: float          # = v_i * s_i * m_i * rel_i


@dataclass
class ScoredArticle:
    #One scored article: the row schema for the scored corpus output
    story_id:          str
    country:           str
    score:             float
    n_hits:            int
    n_unique_terms:    int
    sum_contributions: float
    length_norm:       float
    article_length:    int
    c_o:               float
    mean_severity:     float
    mean_intensity:    float
    mean_relevance:    float
    mean_valence:      float


def _aggregate_multiplicative(per_hit: list[HitDecomposition]) -> float:
    #Default: Σ vᵢ·sᵢ·mᵢ·relᵢ.
    return sum(h.contribution for h in per_hit)


def _aggregate_additive(per_hit: list[HitDecomposition]) -> float:
    '''
    Additive sensitivity variant: preserves valence sign and severity-weighted
    relevance, without letting intensity amplify severity multiplicatively.
    '''
    return sum(
        h.valence * (h.severity + (h.intensity - 1.0) + h.relevance)
        for h in per_hit
    )


def _aggregate_loglinear(per_hit: list[HitDecomposition]) -> float:
    '''
    Log-linear sensitivity variant. Applies sign-preserving log to each
    contribution before summing.
    '''
    total = 0.0
    for h in per_hit:
        c = h.contribution
        total += math.copysign(math.log1p(abs(c)), c)
    return total


AGGREGATORS: dict[str, Callable[[list[HitDecomposition]], float]] = {
    "multiplicative": _aggregate_multiplicative,
    "additive":       _aggregate_additive,
    "loglinear":      _aggregate_loglinear,
}


In [30]:
# ARTICLE-LEVEL SCORE WITH PER-HIT DECOMPOSITION

def score_article(
    doc: Doc,
    country_code: str,
    c_o: float,
    hazard_matcher: PhraseMatcher,
    story_id: str = "",
    alpha: float = ALPHA,
    aggregator: str = "multiplicative",
) -> tuple[ScoredArticle, list[HitDecomposition]]:
    '''
    Compute S(n,t,k) for one article.

    Returns:
      ScoredArticle  : article-level summary
      hits_decomp    : per-hit contributions for audit/inspection
    '''
    if aggregator not in AGGREGATORS:
        raise ValueError(
            f"Unknown aggregator: {aggregator}. Choose from {list(AGGREGATORS)}"
        )

    I_l = max(len(doc), 1)
    length_norm = c_o / (I_l ** alpha)

    hits = find_hazard_hits(doc, hazard_matcher)
    if not hits:
        return (
            ScoredArticle(
                story_id=story_id,
                country=country_code,
                score=0.0,
                n_hits=0,
                n_unique_terms=0,
                sum_contributions=0.0,
                length_norm=length_norm,
                article_length=I_l,
                c_o=c_o,
                mean_severity=np.nan,
                mean_intensity=np.nan,
                mean_relevance=np.nan,
                mean_valence=np.nan,
            ),
            [],
        )

    per_hit: list[HitDecomposition] = []
    for hit_start, hit_end, term in hits:
        s = get_severity(term)
        v = get_valence(doc, hit_start)
        m = get_intensity(doc, hit_start, hit_end)
        r = get_relevance(doc, hit_start, country_code)
        per_hit.append(
            HitDecomposition(
                story_id=story_id,
                country=country_code,
                term=term,
                hit_start=hit_start,
                hit_end=hit_end,
                severity=s,
                valence=v,
                intensity=m,
                relevance=r,
                contribution=v * s * m * r,
            )
        )

    total_contributions = AGGREGATORS[aggregator](per_hit)
    score = length_norm * total_contributions

    summary = ScoredArticle(
        story_id=story_id,
        country=country_code,
        score=score,
        n_hits=len(per_hit),
        n_unique_terms=len({h.term for h in per_hit}),
        sum_contributions=total_contributions,
        length_norm=length_norm,
        article_length=I_l,
        c_o=c_o,
        mean_severity=float(np.mean([h.severity for h in per_hit])),
        mean_intensity=float(np.mean([h.intensity for h in per_hit])),
        mean_relevance=float(np.mean([h.relevance for h in per_hit])),
        mean_valence=float(np.mean([h.valence for h in per_hit])),
    )
    return summary, per_hit


In [31]:
# CORPUS-LEVEL SCORING

def score_corpus(
    corpus_df: pd.DataFrame,
    nlp: spacy.language.Language,
    hazard_matcher: PhraseMatcher | None = None,
    alpha: float = ALPHA,
    aggregator: str = "multiplicative",
) -> tuple[pd.DataFrame, pd.DataFrame]:

    if corpus_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    required = {"story_id", "country", "text", "c_o"}
    missing = required - set(corpus_df.columns)
    if missing:
        raise ValueError(f"corpus_df missing required columns: {missing}")

    if hazard_matcher is None:
        hazard_matcher = build_phrase_matcher(nlp, LEXICON_ALL)

    log.info(
        f"=== PHASE 2 | scoring {len(corpus_df)} articles "
        f"(aggregator='{aggregator}', α={alpha}) ==="
    )

    summaries: list[ScoredArticle] = []
    all_hits: list[HitDecomposition] = []

    texts = corpus_df["text"].fillna("").tolist()
    docs = nlp.pipe(texts, batch_size=256)  # ↑ 64→256: better GPU/CPU utilisation

    for (_, row), doc in zip(corpus_df.iterrows(), docs):
        summary, hits = score_article(
            doc=doc,
            country_code=row["country"],
            c_o=float(row["c_o"]),
            hazard_matcher=hazard_matcher,
            story_id=str(row["story_id"]),
            alpha=alpha,
            aggregator=aggregator,
        )
        summaries.append(summary)
        all_hits.extend(hits)

    scored_df = pd.DataFrame([asdict(s) for s in summaries])
    hits_df = pd.DataFrame([asdict(h) for h in all_hits])

    # Merge auxiliary columns from Phase 1 onto scored_df for downstream use
    aux_cols = [
        c for c in (
            "assigned_date", "publish_dt", "publish_dt_utc", "media_name",
            "category", "cred_tier", "title", "url"
        )
        if c in corpus_df.columns
    ]
    if aux_cols:
        scored_df = scored_df.merge(
            corpus_df[["story_id", *aux_cols]],
            on="story_id",
            how="left",
        )

    log.info(
        f"  scored: {len(scored_df)} articles, {len(hits_df)} total hits "
        f"({hits_df['term'].nunique() if not hits_df.empty else 0} unique terms)"
    )
    log.info(
        f"  score distribution: mean={scored_df['score'].mean():.4f}, "
        f"median={scored_df['score'].median():.4f}, "
        f"max={scored_df['score'].max():.4f}, "
        f"n_zero={(scored_df['score'] == 0).sum()}"
    )

    return scored_df, hits_df


## Phase 3: Novelty Weighting (reused by production)

> **Phase 3 config constants.** 

`EMBEDDING_MODEL`/`NOVELTY_WINDOW_DAYS` are used directly by the production Phase 2 cell's novelty-weighting call 

`SSI_LAMBDA`/`AR_MAX_LAGS`/`CSP_ROLLING_STD_WINDOW` are used directly by the production Phase 3 cell. 

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL         = "all-MiniLM-L6-v2"   # novelty-weight embedding model
NOVELTY_WINDOW_DAYS     = 7                     # prior-days window for novelty comparison
SSI_HALF_LIFE_DAYS      = 30
SSI_LAMBDA              = float(np.exp(-np.log(2) / SSI_HALF_LIFE_DAYS))
AR_MAX_LAGS             = 5                     # BIC-selected AR(p), p in {0,...,5}
CSP_ROLLING_STD_WINDOW  = 252                   # trailing days for CSP standardisation


 **Sentence embeddings + novelty weights.** 

`compute_novelty_weights()` and its embedding loader `get_embed_model()`, called directly inside the production Phase 2 cell's internal `_safe_compute_beta()` helper. 

`compute_daily_score()` is called directly by the production Phase 3 cell.

In [33]:
#  Sentence embeddings for novelty weights 
# Reimers & Gurevych (2019); Wang et al. (2020)

_embed_model_cache: dict[str, SentenceTransformer] = {}

def get_embed_model(model_name: str = EMBEDDING_MODEL) -> SentenceTransformer:
    if model_name not in _embed_model_cache:
        log.info(f"Loading embedding model: {model_name}")
        _embed_model_cache[model_name] = SentenceTransformer(model_name)
    return _embed_model_cache[model_name]

def compute_embeddings(texts: list[str], model_name: str = EMBEDDING_MODEL) -> np.ndarray:
    #Batch-encode texts -> embedding matrix (N × D).
    model = get_embed_model(model_name)
    return model.encode(texts, batch_size=128, show_progress_bar=False, 
                        convert_to_numpy=True, normalize_embeddings=True)


In [34]:
#   Novelty weight w_n computation 

def compute_novelty_weights(df: pd.DataFrame,
                            model_name: str = EMBEDDING_MODEL,
                            window_days: int = NOVELTY_WINDOW_DAYS) -> pd.Series:
    '''
    For each article in df, compute w_n = 1 - max cosine similarity to all
    same-country articles in the prior `window_days` days.

    Articles are processed in chronological order per country.
    Embeddings are computed in batch for efficiency.

    Speed change vs original: the per-row Python loop that computed
    embeddings[prior_idx] @ embeddings[i] one article at a time is replaced
    by a single NumPy matrix multiply over the entire prior window at once.
    Semantics and numerical output are identical.

    Returns pd.Series of w_n values, indexed as df.
    '''
    df = df.copy().sort_values("assigned_date")
    texts      = df["text"].fillna("").tolist()
    embeddings = compute_embeddings(texts, model_name)  # (N, D), L2-normalised

    betas     = np.ones(len(df))
    dates     = df["assigned_date"].values
    window_td = np.timedelta64(window_days, "D")

    for i in range(len(df)):
        mask = (dates < dates[i]) & (dates >= dates[i] - window_td)
        prior_idx = np.where(mask)[0]
        if len(prior_idx) == 0:
            betas[i] = 1.0   
            continue
        # Vectorised: compute all cosine similarities in one matmul.
        # embeddings are L2-normalised so dot product == cosine similarity.
        sims     = embeddings[prior_idx] @ embeddings[i]  # (|H_n|,)
        max_sim  = float(np.max(sims))
        betas[i] = max(0.0, 1.0 - max_sim)  # w_n ∈ [0,1]

    return pd.Series(betas, index=df.index)


In [35]:
#  Daily novelty-weighted score S(k) 

def compute_daily_score(df: pd.DataFrame) -> pd.Series:
    '''
    Compute daily novelty-weighted mean score:
      S(t,k) = Σ_n w_n * S(n,t,k) / Σ_n w_n

    This eliminates mechanical dependence on article volume while correctly
    down-weighting wire-service re-reporting of the same underlying event.

    Returns pd.Series indexed by date.
    '''
    daily = (
        df.groupby("assigned_date")
          .apply(lambda g: (
              np.dot(g["beta"], g["score"]) / g["beta"].sum()
              if g["beta"].sum() > 0 else 0.0
          ))
    )
    return daily.rename("S_daily")


## Full-Sample US Production Run

This is the run that generated the reported thesis results (2008–2025).

**504-error fix.** Splits the query into two halves (Media Cloud backend timeout on the full query). Defines `_gather_corpus_split_query()`, used directly by the Stage A run below. 

 - Read timeout: server never responds (fixed by raising timeout to 180s)
 - 504 Gateway Timeout: server responds but says "my backend timed out"
This means the query itself is too heavy — must use a smaller query

This doubles API calls per week (2 instead of 1) but eliminates 504s.

In [36]:
import time
import types
import math

#  Splits the full hazard query into N balanced sub-queries 

def _split_query_into_halves(full_query: str, n_parts: int = 2) -> list[str]:
    '''
    Split a large OR query into n_parts smaller OR queries.
    Each part contains roughly 1/n_parts of the original terms.
    Input:  '("term a" OR "term b" OR "term c" ...)'
    Output: ['("term a" OR "term b" ...)', '("term c" ...)', ...]
    '''
    inner = full_query.strip()
    if inner.startswith("(") and inner.endswith(")"):
        inner = inner[1:-1]

    terms = [t.strip() for t in inner.split(" OR ")]
    terms = [t for t in terms if t]  # remove empties

    chunk_size = math.ceil(len(terms) / n_parts)
    parts = []
    for i in range(0, len(terms), chunk_size):
        chunk = terms[i:i + chunk_size]
        parts.append("(" + " OR ".join(chunk) + ")")

    return parts


# Build the full hazard query (same as gather_corpus uses internally)
_full_query = build_hazard_query(
    TAXONOMY,
    weather_disaster_only=True,
    require_climate_context=False,
    min_severity_for_query=2,
    category=None,
)

_query_halves = _split_query_into_halves(_full_query, n_parts=2)

print(f"Full query terms:    {_full_query.count(' OR ') + 1}")
print(f"Half 1 terms:        {_query_halves[0].count(' OR ') + 1}")
print(f"Half 2 terms:        {_query_halves[1].count(' OR ') + 1}")
print(f"API calls per week:  2 (was 1)")
print(f"Estimated runtime:   ~3.5 days (was ~2 days with 1 call)")


#  New gather_corpus wrapper that fires both halves and merges 

def _gather_corpus_split_query(
    country_code: str,
    start,
    end,
    mc_client,
    nlp,
    haz_matcher,
    exc_matcher,
    query_parts: list[str],
    category=None,
):
    '''
    Fire gather_corpus once per query half, deduplicate on story_id, return
    the merged result in the same (corpus_df, rejects_df, audit) format.

    Falls back to a single full-query call if a half-query also 504s
    (unlikely since the payload is now half the size).
    '''
    import pandas as pd

    all_corps   = []
    all_rejects = []
    all_audits  = []
    seen_ids    = set()

    for part_n, part_query in enumerate(query_parts, 1):
        log.info(f"  Query part {part_n}/{len(query_parts)}: "
                 f"{part_query.count(' OR ') + 1} terms")

        # Temporarily override the query inside gather_corpus by patching build_hazard_query to return our pre-built part query
        _orig_bhq = globals().get("build_hazard_query")

        def _patched_bhq(*args, **kwargs):
            return part_query

        # Inject the patch into gather_corpus's global namespace
        gather_corpus.__globals__["build_hazard_query"] = _patched_bhq

        try:
            corp, rej, aud = gather_corpus(
                country_code, start, end, mc_client,
                nlp, haz_matcher, exc_matcher,
                category=category,
            )
        except Exception as e:
            log.warning(f"  Part {part_n} failed: {e}")
            corp = pd.DataFrame()
            rej  = pd.DataFrame()
            aud  = []
        finally:
            # restore original build_hazard_query
            if _orig_bhq is not None:
                gather_corpus.__globals__["build_hazard_query"] = _orig_bhq
            else:
                del gather_corpus.__globals__["build_hazard_query"]

        # Deduplicate on story_id 
        if not corp.empty and "story_id" in corp.columns:
            new_rows = corp[~corp["story_id"].isin(seen_ids)]
            seen_ids.update(new_rows["story_id"].tolist())
            if not new_rows.empty:
                all_corps.append(new_rows)
        elif not corp.empty:
            all_corps.append(corp)

        if rej is not None and not rej.empty:
            all_rejects.append(rej)
        if aud:
            all_audits.extend(aud)

        
        time.sleep(SECONDS_BETWEEN_REQS)

    merged_corp = (pd.concat(all_corps,   ignore_index=True, sort=False)
                   if all_corps   else pd.DataFrame())
    merged_rej  = (pd.concat(all_rejects, ignore_index=True, sort=False)
                   if all_rejects else pd.DataFrame())

    return merged_corp, merged_rej, all_audits


print("\n✓ _gather_corpus_split_query defined")

Full query terms:    156
Half 1 terms:        78
Half 2 terms:        78
API calls per week:  2 (was 1)
Estimated runtime:   ~3.5 days (was ~2 days with 1 call)

✓ _gather_corpus_split_query defined


**Stage A — production retrieval.** 2008–2025, weekly checkpointing (resumable), robust HTTP retry/backoff. 

In [37]:
# FULL PERIOD US RUN — STAGE A ONLY

from pathlib import Path
from datetime import date, datetime, timedelta
import os, re, json, logging, time, types
import numpy as np
import pandas as pd
import requests, requests.adapters
from urllib3.util.retry import Retry

try:
    import statsmodels.api as sm
except Exception as e:
    print(f"⚠ statsmodels: {e}")

# CONFIGURATION

RUN_COUNTRY = "US"
RUN_START   = date(2008, 1, 1)
RUN_END     = date(2025, 12, 31)
MAX_WEEKS   = None

USE_SENTENCE_TRANSFORMER_BETA = True


READ_TIMEOUT_S      = 180
CONNECT_TIMEOUT_S   = 15
FAST_DELAY_S        = 20
TIMEOUT_RETRY_WAITS = [90, 180, 300]

# 1. OUTPUT DIRECTORIES


if "FULL_YEAR_RUN_DIR_US" not in globals():
    _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    FULL_YEAR_RUN_DIR_US    = (THESIS_DATA_ROOT
                                / f"full_period_US_{RUN_START}_{RUN_END}_{_ts}")
    FULL_YEAR_STAGEA_DIR_US = FULL_YEAR_RUN_DIR_US / "stageA"
    FULL_YEAR_STAGEA_DIR_US.mkdir(parents=True, exist_ok=True)
    print(f"✓ New output folder: {FULL_YEAR_RUN_DIR_US}")
else:
    print(f"✓ Resuming into existing folder: {FULL_YEAR_RUN_DIR_US}")

# 2. REQUIRED OBJECT CHECKS


if "MEDIA_CLOUD_API_KEY" not in globals() or not MEDIA_CLOUD_API_KEY:
    raise RuntimeError(
        "MEDIA_CLOUD_API_KEY is not set. Run the Setup section above first, "
        "or set the MEDIA_CLOUD_API_KEY environment variable.")

required_objects = [
    "RateLimitedMediaCloud", "gather_corpus", "load_spacy_model",
    "build_phrase_matcher", "LEXICON_ALL", "TRANSITION_EXCLUSION_LEXICON",
    "PHASE1_COUNTRIES",
]
_missing = [x for x in required_objects if x not in globals()]
if _missing:
    raise RuntimeError(
        f"Run all notebook setup cells first. Missing: {_missing}")

if "US" not in PHASE1_COUNTRIES:
    raise RuntimeError("PHASE1_COUNTRIES has no 'US' entry.")
if not PHASE1_COUNTRIES["US"].get("ready", False):
    raise RuntimeError(
        "PHASE1_COUNTRIES['US']['ready'] is False. "
        "Set it to True in the Phase 1 config cell.")

print(f"✓ US config: {PHASE1_COUNTRIES['US']}")
print(f"✓ US source IDs: {mediacloud_source_ids_for('US')}")

if "log" not in globals():
    logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s | %(levelname)s | %(message)s")
    log = logging.getLogger("full_period_us")


# 3. PATCH HTTP LAYER — 180s timeout & smart retry


class _RobustAdapter(requests.adapters.HTTPAdapter):
    def __init__(self):
        _retry = Retry(total=2, connect=2, read=0,
                       backoff_factor=1.5,
                       status_forcelist=[500, 502, 503, 504],
                       raise_on_status=False)
        super().__init__(max_retries=_retry,
                         pool_connections=2, pool_maxsize=4)

    def send(self, *args, **kwargs):
        kwargs["timeout"] = (CONNECT_TIMEOUT_S, READ_TIMEOUT_S)
        return super().send(*args, **kwargs)


def _apply_robust_adapter(client_obj, label: str) -> bool:
    session = None
    for path in [["client", "_session"], ["client", "session"],
                 ["_session"], ["session"]]:
        try:
            obj = client_obj
            for attr in path:
                obj = getattr(obj, attr)
            if isinstance(obj, requests.Session):
                session = obj
                break
        except AttributeError:
            continue
    if session is None:
        return False
    adapter = _RobustAdapter()
    session.mount("https://", adapter)
    session.mount("http://",  adapter)
    session.headers.update({"Connection": "keep-alive"})
    print(f"  ✓ 180s adapter applied: {label}")
    return True


def _robust_story_list(self, query, start_date, end_date,
                        collection_ids=None, source_ids=None,
                        pagination_token=None, page_size=500):
    if source_ids and collection_ids:
        raise ValueError("Pass source_ids OR collection_ids, not both.")
    if not source_ids and not collection_ids:
        raise ValueError("Must pass source_ids or collection_ids.")
    if not query or not str(query).strip():
        raise ValueError("Query is empty.")

    self._check_budget()
    self._wait()

    start_obj = self._date_obj(start_date)
    end_obj   = self._date_obj(end_date)
    clean_src = [int(x) for x in (source_ids     or [])]
    clean_col = [int(x) for x in (collection_ids or [])]

    all_waits = [0] + TIMEOUT_RETRY_WAITS

    for _attempt, _wait in enumerate(all_waits):
        if _wait > 0:
            log.warning(f"  Timeout retry {_attempt}/{len(TIMEOUT_RETRY_WAITS)} "
                        f"({start_date}→{end_date}) — waiting {_wait}s")
            time.sleep(_wait)
            _apply_robust_adapter(self, f"reconnect-{_attempt}")

        try:
            response = self.client.story_list(
                query=query,
                start_date=start_obj,
                end_date=end_obj,
                collection_ids=clean_col,
                source_ids=clean_src,
                platform="onlinenews-mediacloud",
                pagination_token=pagination_token,
                page_size=page_size,
            )
            self.budget.increment(self.budget_file)
            self._last_req = time.monotonic()
            return self._normalise_story_response(response)

        except Exception as _e:
            _is_retryable = any(kw in str(_e).lower() for kw in (
                "timed out", "timeout", "read timeout",
                "connectionreset", "remotedisconnected",
                "504", "502", "503", "429",
                "bad gateway", "gateway timeout", "service unavailable",
                "too many requests",
            ))
            if not _is_retryable:
                raise
            if _attempt >= len(TIMEOUT_RETRY_WAITS):
                log.error(f"  All retries exhausted for {start_date}→{end_date}.")
                return [], None
            log.warning(f"  Retryable error (attempt {_attempt}): {_e}")

    return [], None


# 4. INITIALISE CLIENT & NLP TOOLS


_api_key = MEDIA_CLOUD_API_KEY

mc_us = RateLimitedMediaCloud(
    _api_key,
    Path(".mediacloud_budget_US_full_period.json")
)
_apply_robust_adapter(mc_us, "mc_us")
mc_us.story_list = types.MethodType(_robust_story_list, mc_us)

_old_delay = globals().get("SECONDS_BETWEEN_REQS", 33)
SECONDS_BETWEEN_REQS = FAST_DELAY_S
print(f"  ✓ Between-request delay: {_old_delay}s → {SECONDS_BETWEEN_REQS}s")
print(f"  ✓ Budget remaining: {mc_us.budget.remaining():,}")

if "nlp" not in globals():
    print("Loading spaCy model...")
    nlp = load_spacy_model()
    nlp.max_length = 2_000_000
    print(f"  ✓ spaCy: {nlp.meta.get('name','?')}")

hazard_matcher    = build_phrase_matcher(nlp, LEXICON_ALL,
                                          label="HAZARD", attr="LOWER")
exclusion_matcher = build_phrase_matcher(nlp, TRANSITION_EXCLUSION_LEXICON,
                                          label="EXCLUSION", attr="LOWER")
print(f"  ✓ Hazard matcher: {len(LEXICON_ALL):,} terms")


# 5. HELPER FUNCTIONS


def _safe_to_csv(df: pd.DataFrame, path: Path) -> Path:
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].astype(str)
        elif out[col].dtype == "object":
            out[col] = out[col].apply(
                lambda x: json.dumps(x, default=str)
                if isinstance(x, (dict, list, tuple, set)) else x)
    path.parent.mkdir(parents=True, exist_ok=True) 
    out.to_csv(path, index=False, encoding="utf-8-sig")
    return path

def _drop_nonserializable(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[c for c in
        ["doc", "country_mentions", "diagnostic_taxonomy_terms"]
        if c in df.columns], errors="ignore").copy()

def _week_ranges(start: date, end: date):
    cur = start
    while cur <= end:
        yield cur, min(cur + timedelta(days=6), end)
        cur += timedelta(days=7)

def _ensure_story_id(df: pd.DataFrame, prefix: str = "row") -> pd.DataFrame:
    out = df.copy()
    if "story_id" not in out.columns:
        out["story_id"] = [f"{prefix}_{i}" for i in range(len(out))]
    out["story_id"] = out["story_id"].fillna("").astype(str)
    blank = out["story_id"].str.strip() == ""
    if blank.any():
        out.loc[blank, "story_id"] = [f"{prefix}_{i}" for i in out.index[blank]]
    if out["story_id"].duplicated().any():
        out["story_id"] = (out["story_id"].astype(str) + "_"
                           + out.groupby("story_id").cumcount().astype(str))
    return out

def _normalize_pipeline_dates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "assigned_date" not in out.columns:
        for col in ("publish_dt_utc", "publish_dt", "market_day"):
            if col in out.columns:
                out["assigned_date"] = out[col]; break
        else:
            out["assigned_date"] = pd.Timestamp(RUN_START)
    out["assigned_date"] = pd.to_datetime(out["assigned_date"], errors="coerce")
    out.loc[out["assigned_date"].isna(), "assigned_date"] = pd.Timestamp(RUN_START)
    out["assigned_date"] = out["assigned_date"].dt.normalize()
    return out

def _week_already_done(ws: date) -> bool:
    p = FULL_YEAR_STAGEA_DIR_US / f"US_{ws}_kept.csv"
    return p.exists() and p.stat().st_size > 100

def _write_week_checkpoint(df: pd.DataFrame, ws: date) -> None:
    out = _drop_nonserializable(df) if not df.empty else pd.DataFrame(
        columns=["story_id", "country", "validation_week_start"])
    if not df.empty:
        out["validation_week_start"] = str(ws)
    FULL_YEAR_STAGEA_DIR_US.mkdir(parents=True, exist_ok=True)  
    out.to_csv(FULL_YEAR_STAGEA_DIR_US / f"US_{ws}_kept.csv",
               index=False, encoding="utf-8-sig")


# 6. PHASE 1 STAGE A — weekly corpus collection with checkpointing


print("\n" + "═" * 100)
print(f"PHASE 1 — STAGE A  |  {RUN_COUNTRY}  |  {RUN_START} → {RUN_END}")
print("═" * 100)
print("NOTE: Per-week checkpointing enabled — re-run this cell to resume")

_all_weeks = list(_week_ranges(RUN_START, RUN_END))
if MAX_WEEKS:
    _all_weeks = _all_weeks[:int(MAX_WEEKS)]

_total   = len(_all_weeks)
_todo    = [(_ws, _we) for _ws, _we in _all_weeks if not _week_already_done(_ws)]
_skipped = _total - len(_todo)

print(f"\nTotal weeks: {_total}  |  Already done: {_skipped}  |  To fetch: {len(_todo)}")
print(f"Estimated time: ~{len(_todo) * 110 / 3600:.1f} hours\n")

stageA_frames        = []
stageA_reject_frames = []
stageA_audit_records = []
stageA_week_summary  = []
_done_count          = _skipped
_last_10_times       = []

for _ws, _we in _todo:
    _done_count += 1
    _t0 = time.time()

    if _last_10_times:
        _eta_h   = (_total - _done_count) * (sum(_last_10_times) / len(_last_10_times)) / 3600
        _eta_str = f"  ETA ~{_eta_h:.1f}h"
    else:
        _eta_str = ""

    print(f"\n[{_done_count}/{_total}] {_ws} → {_we}{_eta_str}")

    time.sleep(SECONDS_BETWEEN_REQS)

    try:
        corpus_df, rejects_df, audit = _gather_corpus_split_query(
            RUN_COUNTRY, _ws, _we,
            mc_us, nlp, hazard_matcher, exclusion_matcher,
            query_parts=_query_halves,
            category=None,
        )

        corpus_df = _ensure_story_id(corpus_df, prefix=f"US_{_ws}")
        corpus_df = _normalize_pipeline_dates(corpus_df)

        if "country" not in corpus_df.columns and not corpus_df.empty:
            corpus_df["country"] = RUN_COUNTRY

        if not corpus_df.empty:
            corpus_df["validation_country"]    = RUN_COUNTRY
            corpus_df["validation_week_start"] = str(_ws)
            stageA_frames.append(corpus_df)

        if rejects_df is not None and not rejects_df.empty:
            rejects_df = rejects_df.copy()
            rejects_df["validation_country"]    = RUN_COUNTRY
            rejects_df["validation_week_start"] = str(_ws)
            stageA_reject_frames.append(rejects_df)
            _safe_to_csv(
                _drop_nonserializable(rejects_df),
                FULL_YEAR_STAGEA_DIR_US / f"US_{_ws}_stageA_rejects.csv"
            )

        if audit:
            stageA_audit_records.extend(audit)

        _write_week_checkpoint(corpus_df, _ws)

        _elapsed = time.time() - _t0
        _last_10_times = (_last_10_times + [_elapsed])[-10:]

        stageA_week_summary.append({
            "country": RUN_COUNTRY, "week_start": _ws, "week_end": _we,
            "stageA_kept":    len(corpus_df),
            "stageA_rejects": len(rejects_df) if rejects_df is not None else 0,
            "audit_records":  len(audit) if audit else 0,
            "elapsed_s":      round(_elapsed, 1),
            "error": "", "status": "ok"
        })

        print(f"  Kept:    {len(corpus_df):,}")
        print(f"  Rejects: {len(rejects_df) if rejects_df is not None else 0:,}")
        print(f"  Audit:   {len(audit) if audit else 0:,}")
        print(f"  Time:    {_elapsed:.0f}s")

    except Exception as _e:
        _elapsed = time.time() - _t0
        print(f"  ✗ Failed: {type(_e).__name__}: {str(_e)[:120]}")
        stageA_week_summary.append({
            "country": RUN_COUNTRY, "week_start": _ws, "week_end": _we,
            "stageA_kept": 0, "stageA_rejects": 0, "audit_records": 0,
            "elapsed_s": round(_elapsed, 1),
            "error": f"{type(_e).__name__}: {str(_e)[:300]}",
            "status": "failed"
        })
        _write_week_checkpoint(pd.DataFrame(), _ws)
        time.sleep(120)

    _safe_to_csv(
        pd.DataFrame(stageA_week_summary),
        FULL_YEAR_STAGEA_DIR_US / "US_stageA_week_summary.csv"
    )

#  Combines all weeks from disk 

_all_kept_files = sorted(FULL_YEAR_STAGEA_DIR_US.glob("US_*_kept.csv"))
_reload_frames  = []
for _f in _all_kept_files:
    try:
        _df = pd.read_csv(_f)
        if not _df.empty and "story_id" in _df.columns:
            _reload_frames.append(_df)
    except Exception:
        continue

full_year_stageA = (pd.concat(_reload_frames, ignore_index=True, sort=False)
                    if _reload_frames else pd.DataFrame())

if not full_year_stageA.empty and "story_id" in full_year_stageA.columns:
    _before = len(full_year_stageA)
    full_year_stageA = full_year_stageA.drop_duplicates(
        subset=["story_id"], keep="first")
    if len(full_year_stageA) < _before:
        print(f"  Cross-week dedup: {_before:,} → {len(full_year_stageA):,}")

stageA_summary_df = pd.DataFrame(stageA_week_summary)

_safe_to_csv(_drop_nonserializable(full_year_stageA),
             FULL_YEAR_STAGEA_DIR_US / "US_full_period_stageA_combined.csv")
_safe_to_csv(stageA_summary_df,
             FULL_YEAR_STAGEA_DIR_US / "US_full_period_stageA_week_summary.csv")

print(f"\n{'═' * 100}")
print("✓ Stage A complete")
print(f"  Total kept articles : {len(full_year_stageA):,}")
print(f"  Output folder       : {FULL_YEAR_STAGEA_DIR_US}")
print(f"  Combined CSV        : {FULL_YEAR_STAGEA_DIR_US / 'US_full_period_stageA_combined.csv'}")
print(f"  Week summary        : {FULL_YEAR_STAGEA_DIR_US / 'US_full_period_stageA_week_summary.csv'}")

display(stageA_summary_df.tail(10))

if full_year_stageA.empty:
    raise RuntimeError(
        "Stage A produced no articles.\n"
        "Check: 1) PHASE1_COUNTRIES['US']['ready'] is True\n"
        "       2) mediacloud_source_ids_for('US') returns a non-empty list\n"
        "       3) Budget not exhausted (run the budget check cell)\n"
        "       4) Week summary CSV for per-week errors"
    )

✓ New output folder: /Users/adam/Desktop/PCRA Index Thesis/Data Gathering/thesis_data/full_period_US_2008-01-01_2025-12-31_20260811_203350
✓ US config: {'collection_ids': [34412234], 'tz_name': 'America/New_York', 'ready': True}
✓ US source IDs: [1089, 1, 4, 1752, 1095, 162705, 25499]
  ✓ 180s adapter applied: mc_us
  ✓ Between-request delay: 33.0s → 20s
  ✓ Budget remaining: 4,000
Loading spaCy model...


2026-08-11 20:33:51,254 | INFO | spaCy loaded: en_core_web_trf


  ✓ spaCy: core_web_trf
  ✓ Hazard matcher: 341 terms

════════════════════════════════════════════════════════════════════════════════════════════════════
PHASE 1 — STAGE A  |  US  |  2008-01-01 → 2025-12-31
════════════════════════════════════════════════════════════════════════════════════════════════════
NOTE: Per-week checkpointing enabled — re-run this cell to resume

Total weeks: 940  |  Already done: 0  |  To fetch: 940
Estimated time: ~28.7 hours


[1/940] 2008-01-01 → 2008-01-07


KeyboardInterrupt: 

**Pre-Step-8 date fix.** 

Converts date-like string columns to datetime before Stage B (needed because syndication cleanup does datetime arithmetic).

Merge all data runs if needed otherwise proceed

In [38]:
# MERGE ALL STAGE A RUNS -> FINAL CLEAN PHASE 1 CORPUS
#
# MODE = 1 : auto-discover every Stage A output folder under THESIS_DATA_ROOT
#            (matching "full_period_US_<start>_<end>_<timestamp>" naming),
#            merge all their per-week US_<date>_kept.csv files, dedup.
# MODE = 2 : load directly from the single canonical target folder
#            (ACTUAL_RUN_ROOT / "Actual Run Data" / "Phase 1 - Articles Kept
#            vs Rejected" / "Stage A" / "Full Data") containing all weekly
#            kept.csv files (2008–2025). Path resolution is whitespace-safe
#            (ignores stray leading/trailing spaces in folder names).
#
# Output: full_year_stageA, empty_weeks_df, missing_weeks_df


from pathlib import Path
import pandas as pd

MODE = 2   # <-- set to 1 (discover + merge multiple runs) or 2 (load single target folder)

def find_child(parent: Path, target_name: str) -> Path:
    """Find a subfolder matching target_name, ignoring leading/trailing whitespace."""
    matches = [p for p in parent.iterdir()
               if p.is_dir() and p.name.strip() == target_name.strip()]
    if not matches:
        available = [repr(p.name) for p in parent.iterdir() if p.is_dir()]
        raise RuntimeError(
            f"No folder matching {repr(target_name)} found under {parent}.\n"
            f"Available subfolders: {available}"
        )
    return matches[0]

# MODE 1 
if MODE == 1:
    RUN_DIRS = sorted(
        d / "stageA"
        for d in THESIS_DATA_ROOT.glob("full_period_US_*")
        if (d / "stageA").is_dir()
    )

    print(f"[MODE 1] Discovered {len(RUN_DIRS)} Stage A run folder(s) under {THESIS_DATA_ROOT}:")
    for d in RUN_DIRS:
        print(f"  {d}")

    if not RUN_DIRS:
        raise RuntimeError(
            f"No Stage A output folders found under {THESIS_DATA_ROOT}. "
            "Check THESIS_DATA_ROOT is set correctly and at least one "
            "'full_period_US_*/stageA' folder exists."
        )

    run_frames   = []
    load_summary = []
    empty_weeks  = []

    for run_idx, folder in enumerate(RUN_DIRS):
        kept_files = sorted(folder.glob("US_*_kept.csv"))
        print(f"\nRun {run_idx+1}: {folder}")
        print(f"  Found {len(kept_files)} kept-week files")

        for f in kept_files:
            week_str = f.stem.replace("US_", "").replace("_kept", "")
            try:
                df = pd.read_csv(f)
                if df.empty or "story_id" not in df.columns:
                    reason = "empty file" if df.empty else "missing story_id column"
                    print(f"  ⚠ {f.name}: 0 rows kept ({reason}) — recording as zero week")
                    empty_weeks.append({"run": run_idx + 1, "week": week_str,
                                         "rows": 0, "file": f.name, "reason": reason})
                    load_summary.append({"run": run_idx + 1, "week": week_str,
                                          "rows": 0, "file": f.name})
                    continue
                df["_source_run"]  = run_idx + 1
                df["_source_file"] = f.name
                run_frames.append(df)
                load_summary.append({
                    "run": run_idx + 1, "week": week_str,
                    "rows": len(df), "file": f.name
                })
            except Exception as e:
                print(f"  ⚠ {f.name}: could not read ({e}) — recording as zero week")
                empty_weeks.append({"run": run_idx + 1, "week": week_str,
                                     "rows": 0, "file": f.name, "reason": str(e)})
                load_summary.append({"run": run_idx + 1, "week": week_str,
                                      "rows": 0, "file": f.name})

    if not run_frames:
        raise RuntimeError(
            "No non-empty kept-week CSVs found across any discovered folder. "
            "Check that at least one Stage A run actually produced articles."
        )

    load_summary_df = pd.DataFrame(load_summary)
    print(f"\nTotal week-files loaded : {len(load_summary_df)}")
    print(f"  Weeks with data       : {(load_summary_df['rows'] > 0).sum()}")
    print(f"  Weeks with zero rows  : {(load_summary_df['rows'] == 0).sum()}")
    print(f"Total rows before any dedup: {load_summary_df['rows'].sum():,}")

    print("\n── Overlap check ──")
    week_run_counts = (load_summary_df.groupby("week")["run"]
                       .nunique().rename("n_runs"))
    overlapping = week_run_counts[week_run_counts > 1]

    if overlapping.empty:
        print("  ✓ No overlapping weeks across runs — safe to merge")
    else:
        print(f"  ⚠ {len(overlapping)} weeks appear in more than one run:")
        print(overlapping.to_string())
        print("  These will be deduplicated by story_id (first occurrence kept)")

    full_year_stageA = pd.concat(run_frames, ignore_index=True, sort=False)
    _before = len(full_year_stageA)

    full_year_stageA = full_year_stageA.drop_duplicates(
        subset=["story_id"], keep="first")
    _after = len(full_year_stageA)

    if _before != _after:
        print(f"\n  Dedup removed {_before - _after:,} duplicate story_ids "
              f"({_before:,} → {_after:,})")
    else:
        print(f"\n  ✓ No duplicate story_ids found — all {_after:,} rows are unique")

#MODE 2 
elif MODE == 2:
    _step = find_child(ACTUAL_RUN_ROOT, "Actual Run Data")
    _step = find_child(_step, "Phase 1 - Articles Kept vs Rejected")
    _step = find_child(_step, "Stage A")
    STAGE_A_DIR = find_child(_step, "Full Data")

    print(f"[MODE 2] Loading all weekly kept-files directly from:")
    print(f"  {STAGE_A_DIR}")

    kept_files = sorted(STAGE_A_DIR.glob("US_*_kept.csv"))
    print(f"  Found {len(kept_files)} kept-week files")

    if not kept_files:
        raise RuntimeError(
            f"No US_*_kept.csv files found in {STAGE_A_DIR}. "
            "Check the folder actually contains the weekly Stage A outputs."
        )

    run_frames   = []
    load_summary = []
    empty_weeks  = []

    for f in kept_files:
        week_str = f.stem.replace("US_", "").replace("_kept", "")
        try:
            df = pd.read_csv(f)
            if df.empty or "story_id" not in df.columns:
                reason = "empty file" if df.empty else "missing story_id column"
                print(f"  ⚠ {f.name}: 0 rows kept ({reason}) — recording as zero week")
                empty_weeks.append({"week": week_str, "rows": 0, "file": f.name, "reason": reason})
                load_summary.append({"week": week_str, "rows": 0, "file": f.name})
                continue
            df["_source_file"] = f.name
            run_frames.append(df)
            load_summary.append({"week": week_str, "rows": len(df), "file": f.name})
        except Exception as e:
            print(f"  ⚠ {f.name}: could not read ({e}) — recording as zero week")
            empty_weeks.append({"week": week_str, "rows": 0, "file": f.name, "reason": str(e)})
            load_summary.append({"week": week_str, "rows": 0, "file": f.name})

    if not run_frames:
        raise RuntimeError(
            "No non-empty kept-week CSVs could be loaded. "
            "Check the files actually contain data."
        )

    load_summary_df = pd.DataFrame(load_summary)
    print(f"\nTotal week-files found : {len(load_summary_df)}")
    print(f"  Weeks with data       : {(load_summary_df['rows'] > 0).sum()}")
    print(f"  Weeks with zero rows  : {(load_summary_df['rows'] == 0).sum()}")
    print(f"Total rows before dedup: {load_summary_df['rows'].sum():,}")

    full_year_stageA = pd.concat(run_frames, ignore_index=True, sort=False)
    _before = len(full_year_stageA)

    full_year_stageA = full_year_stageA.drop_duplicates(
        subset=["story_id"], keep="first")
    _after = len(full_year_stageA)

    if _before != _after:
        print(f"\n  Dedup removed {_before - _after:,} duplicate story_ids "
              f"({_before:,} → {_after:,})")
    else:
        print(f"\n  ✓ No duplicate story_ids found — all {_after:,} rows are unique")

else:
    raise ValueError(f"MODE must be 1 or 2, got {MODE}")

if empty_weeks:
    print(f"\n── Zero-row weeks (timeline preserved, no articles) ──")
    for w in sorted(empty_weeks, key=lambda x: x["week"]):
        print(f"  {w['week']}: {w['reason']}")

empty_weeks_df = pd.DataFrame(empty_weeks)

#  GAP CHECK: any real jump between consecutive found weeks? 
# Rather than assume one fixed weekday-anchor for the whole 2008–2025 span
# (which produced false positives once the anchor day shifted from ~2018
# onward), this checks actual spacing between consecutive found week-files.
# A gap is flagged only if the distance to the next file exceeds ~8 days (allowing 1 day of slack for weekday drift)

print("\n── Gap check: missing week-files (based on actual file spacing) ──")

_week_dates = pd.to_datetime(load_summary_df["week"], errors="coerce")
_week_dates = _week_dates.dropna().sort_values().reset_index(drop=True)

if _week_dates.empty:
    print("  ⚠ Could not parse week dates from filenames — skipping gap check")
    missing_weeks_df = pd.DataFrame()
else:
    _diffs = _week_dates.diff().dt.days
    _gap_threshold = 8  # allow normal 7-day spacing + 1 day slack for anchor drift

    _gap_mask = _diffs > _gap_threshold
    _gap_rows = []

    for idx in _week_dates.index[_gap_mask]:
        prev_date = _week_dates.iloc[idx - 1]
        curr_date = _week_dates.iloc[idx]
        gap_days  = (curr_date - prev_date).days
        _gap_rows.append({
            "gap_start_after": prev_date.date(),
            "gap_end_before":  curr_date.date(),
            "days_missing":    gap_days
        })

    missing_weeks_df = pd.DataFrame(_gap_rows)

    print(f"  Date range covered by files : {_week_dates.min().date()} → {_week_dates.max().date()}")
    print(f"  Total week-files found      : {len(_week_dates)}")
    print(f"  Genuine gaps (>{_gap_threshold} days between files): {len(missing_weeks_df)}")

    if not missing_weeks_df.empty:
        print("\n  Gaps found:")
        for _, row in missing_weeks_df.iterrows():
            print(f"    {row['gap_start_after']} → {row['gap_end_before']} "
                  f"({row['days_missing']} days)")
    else:
        print("  ✓ No genuine gaps — timeline is continuous (allowing for weekday-anchor drift)")

    missing_weeks_df = pd.DataFrame({"missing_week": [d.date() for d in _missing]})

print(f"\n✓ full_year_stageA loaded — {len(full_year_stageA):,} rows (MODE={MODE})")
print(f"✓ empty_weeks_df   — {len(empty_weeks_df)} zero-row/unreadable weeks recorded")

[MODE 2] Loading all weekly kept-files directly from:
  /Users/adam/Desktop/PCRA Index Thesis/Actual Run Data/Phase 1 - Articles Kept vs Rejected/Stage A/Full Data
  Found 940 kept-week files
  ⚠ US_2008-01-08_kept.csv: 0 rows kept (empty file) — recording as zero week
  ⚠ US_2008-01-15_kept.csv: 0 rows kept (empty file) — recording as zero week
  ⚠ US_2008-02-12_kept.csv: 0 rows kept (empty file) — recording as zero week
  ⚠ US_2008-02-19_kept.csv: 0 rows kept (empty file) — recording as zero week
  ⚠ US_2008-02-26_kept.csv: 0 rows kept (empty file) — recording as zero week
  ⚠ US_2008-03-11_kept.csv: 0 rows kept (empty file) — recording as zero week
  ⚠ US_2008-03-25_kept.csv: 0 rows kept (empty file) — recording as zero week
  ⚠ US_2008-11-25_kept.csv: 0 rows kept (empty file) — recording as zero week
  ⚠ US_2009-07-14_kept.csv: 0 rows kept (empty file) — recording as zero week
  ⚠ US_2012-02-14_kept.csv: 0 rows kept (empty file) — recording as zero week

Total week-files found : 94

In [39]:

# PRE-STEP 8 DATE FIX + FINAL SORT/SUMMARY
# Date-column fix only runs for MODE 1 (multi-run merges can pick up mixed
# dtypes/timezones across runs). 
# MODE 2 data comes from one clean folder, so it's assumed already consistent and skips straight to sort/summary.

if MODE == 1:
    print("Pre-processing date columns before Step 8...\n")

    for col in ["publish_dt_utc", "market_day", "validation_week_start"]:
        if col in full_year_stageA.columns:
            full_year_stageA[col] = pd.to_datetime(
                full_year_stageA[col], errors="coerce", utc=False)
            print(f"  ✓ {col} → datetime  "
                  f"(nulls: {full_year_stageA[col].isna().sum():,})")

    if "publish_dt_utc" in full_year_stageA.columns:
        if hasattr(full_year_stageA["publish_dt_utc"].dtype, "tz"):
            full_year_stageA["publish_dt_utc"] = (
                full_year_stageA["publish_dt_utc"].dt.tz_localize(None))
            print("  ✓ publish_dt_utc timezone stripped")

    print("\nDtypes after fix:")
    for col in ["publish_dt_utc", "market_day", "validation_week_start", "assigned_date"]:
        if col in full_year_stageA.columns:
            print(f"  {col:<30} {str(full_year_stageA[col].dtype)}")
else:
    print(f"[MODE {MODE}] Skipping date fix (assumed already clean) — sorting only.\n")

#  SORT BY DATE 

if "assigned_date" in full_year_stageA.columns:
    full_year_stageA["assigned_date"] = pd.to_datetime(
        full_year_stageA["assigned_date"], errors="coerce")
    full_year_stageA = full_year_stageA.sort_values(
        "assigned_date").reset_index(drop=True)

full_year_stageA = full_year_stageA.drop(
    columns=["_source_run", "_source_file"], errors="ignore")

#  COVERAGE SUMMARY 

print("\n── Coverage summary ──")
print(f"  Total articles : {len(full_year_stageA):,}")
if "assigned_date" in full_year_stageA.columns:
    print(f"  Date range     : {full_year_stageA['assigned_date'].min().date()} "
          f"→ {full_year_stageA['assigned_date'].max().date()}")
if "validation_week_start" in full_year_stageA.columns:
    n_weeks = full_year_stageA["validation_week_start"].nunique()
    print(f"  Unique weeks   : {n_weeks}")

print("\n✓ full_year_stageA ready — pass directly to Stage B cell")

[MODE 2] Skipping date fix (assumed already clean) — sorting only.


── Coverage summary ──
  Total articles : 54,394
  Date range     : 2008-01-01 → 2025-12-31
  Unique weeks   : 930

✓ full_year_stageA ready — pass directly to Stage B cell


**Stage B — production run.** Syndication cleanup only (no credibility weighting), self-contained.

- Input:  full_year_stageA   (from merge cell)
- Output: full_year_stageB
- Self-contained — all helpers defined internally

In [40]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

#  Output folder 

MODE = 2   # <-- set to 1 (auto-generate new timestamped folder) or 2 (use Actual Data Run target folder)

if "FULL_YEAR_STAGEB_DIR_US" not in globals():
    if MODE == 1:
        _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        FULL_YEAR_STAGEB_DIR_US = (THESIS_DATA_ROOT
                                    / f"full_period_US_stageB_{_ts}")
        FULL_YEAR_STAGEB_DIR_US.mkdir(parents=True, exist_ok=True)
        print(f"[MODE 1] New Stage B folder: {FULL_YEAR_STAGEB_DIR_US}")
    elif MODE == 2:
        FULL_YEAR_STAGEB_DIR_US = (THESIS_DATA_ROOT
                                    / "Actual Data Run" / "Stage B")
        FULL_YEAR_STAGEB_DIR_US.mkdir(parents=True, exist_ok=True)
        print(f"[MODE 2] Using target Stage B folder: {FULL_YEAR_STAGEB_DIR_US}")
    else:
        raise ValueError(f"MODE must be 1 or 2, got {MODE}")
else:
    print(f"✓ Using existing Stage B folder: {FULL_YEAR_STAGEB_DIR_US}")

#  Guards 

if "full_year_stageA" not in globals() or full_year_stageA.empty:
    raise RuntimeError(
        "full_year_stageA is missing or empty. Run the merge cell first.")

if "RUN_COUNTRY" not in globals():
    RUN_COUNTRY = "US"

if "RUN_START" not in globals():
    from datetime import date
    RUN_START = date(2008, 1, 1)

#  Helpers (self-contained) 

def _safe_to_csv(df: pd.DataFrame, path: Path) -> Path:
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].astype(str)
        elif out[col].dtype == "object":
            out[col] = out[col].apply(
                lambda x: json.dumps(x, default=str)
                if isinstance(x, (dict, list, tuple, set)) else x)
    out.to_csv(path, index=False, encoding="utf-8-sig")
    return path

def _safe_to_parquet(df: pd.DataFrame, path: Path, label: str = ""):
    try:
        df.to_parquet(path, index=False)
        print(f"  ✓ Parquet saved: {label}")
    except Exception as e:
        print(f"  ⚠ Parquet skipped ({label}): {str(e)[:80]}")

def _drop_nonserializable(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[c for c in
        ["doc", "country_mentions", "diagnostic_taxonomy_terms"]
        if c in df.columns], errors="ignore").copy()

def _ensure_story_id(df: pd.DataFrame, prefix: str = "row") -> pd.DataFrame:
    out = df.copy()
    if "story_id" not in out.columns:
        out["story_id"] = [f"{prefix}_{i}" for i in range(len(out))]
    out["story_id"] = out["story_id"].fillna("").astype(str)
    blank = out["story_id"].str.strip() == ""
    if blank.any():
        out.loc[blank, "story_id"] = [f"{prefix}_{i}" for i in out.index[blank]]
    if out["story_id"].duplicated().any():
        out["story_id"] = (out["story_id"].astype(str) + "_"
                           + out.groupby("story_id").cumcount().astype(str))
    return out

def _normalize_pipeline_dates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "assigned_date" not in out.columns:
        for col in ("publish_dt_utc", "publish_dt", "market_day"):
            if col in out.columns:
                out["assigned_date"] = out[col]; break
        else:
            out["assigned_date"] = pd.Timestamp(RUN_START)
    out["assigned_date"] = pd.to_datetime(out["assigned_date"], errors="coerce")
    out.loc[out["assigned_date"].isna(), "assigned_date"] = pd.Timestamp(RUN_START)
    out["assigned_date"] = out["assigned_date"].dt.normalize()
    return out

def _fix_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    #Fix columns that should be numeric but were read as strings from CSV.
    out = df.copy()
    numeric_hints = ["score", "weighted_score", "beta", "c_o",
                     "hazard_count", "word_count", "char_count",
                     "severity", "intensity", "relevance"]
    for col in out.columns:
        if col in numeric_hints:
            out[col] = pd.to_numeric(out[col], errors="coerce")
        elif out[col].dtype == "object":
            try:
                converted = pd.to_numeric(out[col], errors="raise")
                out[col] = converted
            except (ValueError, TypeError):
                pass
    return out



print("\n" + "═" * 100)
print("PHASE 1 — STAGE B  |  Syndication cleanup")
print("═" * 100)
print(f"  Input articles : {len(full_year_stageA):,}")
print(f"  Columns        : {list(full_year_stageA.columns)}")

#  Fix dtypes before passing to cleaner 

print("\nFixing column dtypes...")
full_year_stageA = _fix_numeric_columns(full_year_stageA)
full_year_stageA = _normalize_pipeline_dates(full_year_stageA)
full_year_stageA = _ensure_story_id(full_year_stageA, "US_stageA")
full_year_stageA["publish_dt_utc"] = pd.to_datetime(
    full_year_stageA["publish_dt_utc"], utc=True, errors="coerce"
)
print("  ✓ Dtypes fixed")

#  Fit syndication cleaner 

print("\nFitting syndication cleaner...")
_texts = full_year_stageA["text"].fillna("").astype(str)
_texts = _texts[_texts.str.len() >= 50].tolist()
print(f"  Texts for fitting : {len(_texts):,}")

full_year_cleaner = None
if len(_texts) >= 2:
    full_year_cleaner = SyndicationCleaner()
    full_year_cleaner.fit(_texts)
    try:
        full_year_cleaner.save(
            FULL_YEAR_STAGEB_DIR_US / "syndication_cleaner_US.joblib")
        print("  ✓ Cleaner saved")
    except Exception as e:
        print(f"  ⚠ Cleaner save failed: {e}")
else:
    print("  ⚠ Too few texts to fit cleaner — skipping syndication cleanup")

#  Apply Step 8 

print("\nApplying Step 8 syndication cleanup...")
if full_year_cleaner is not None:
    try:
        full_year_stageB, dropped_story_ids = step8_syndication_cleanup(
            full_year_stageA, full_year_cleaner)
        print(f"  ✓ Step 8 complete")
    except Exception as e:
        print(f"  ⚠ Step 8 failed: {e}")
        print(f"  Keeping all Stage A rows.")
        full_year_stageB  = full_year_stageA.copy()
        dropped_story_ids = set()
else:
    full_year_stageB  = full_year_stageA.copy()
    dropped_story_ids = set()

#  Finalise 

full_year_stageB = _ensure_story_id(full_year_stageB, "US_stageB")
full_year_stageB = _normalize_pipeline_dates(full_year_stageB)
if "country" not in full_year_stageB.columns:
    full_year_stageB["country"] = RUN_COUNTRY

#  Save outputs 

_csv_path     = FULL_YEAR_STAGEB_DIR_US / "US_full_period_stageB_combined.csv"
_parquet_path = FULL_YEAR_STAGEB_DIR_US / "US_full_period_stageB_combined.parquet"

_safe_to_csv(_drop_nonserializable(full_year_stageB), _csv_path)
_safe_to_parquet(_drop_nonserializable(full_year_stageB), _parquet_path, "Stage B")

combined_stageB     = full_year_stageB.copy()
combined_stageB_csv = _drop_nonserializable(full_year_stageB).copy()

print(f"\n✓ Stage B complete")
print(f"  Stage A : {len(full_year_stageA):,}")
print(f"  Dropped : {len(dropped_story_ids):,} syndicated")
print(f"  Stage B : {len(full_year_stageB):,}")
print(f"  CSV     : {_csv_path}")
print(f"  Parquet : {_parquet_path}")

[MODE 2] Using target Stage B folder: /Users/adam/Desktop/PCRA Index Thesis/Data Gathering/thesis_data/Actual Data Run/Stage B

════════════════════════════════════════════════════════════════════════════════════════════════════
PHASE 1 — STAGE B  |  Syndication cleanup
════════════════════════════════════════════════════════════════════════════════════════════════════
  Input articles : 54,394
  Columns        : ['story_id', 'country', 'url', 'title', 'media_name', 'publish_dt_utc', 'market_day', 'assigned_date', 'text', 'text_chars', 'scrape_status', 'retrieval_source', 'is_guardian_api', 'hazard_hits_n', 'hazard_terms', 'validation_country', 'validation_week_start']

Fixing column dtypes...
  ✓ Dtypes fixed

Fitting syndication cleaner...
  Texts for fitting : 54,394
  ✓ Cleaner saved

Applying Step 8 syndication cleanup...


2026-08-11 20:41:06,875 | INFO |   Step 8: 54394 → 41920 after syndication cleanup (12474 dropped)


  ✓ Step 8 complete
  ✓ Parquet saved: Stage B

✓ Stage B complete
  Stage A : 54,394
  Dropped : 12,474 syndicated
  Stage B : 41,920
  CSV     : /Users/adam/Desktop/PCRA Index Thesis/Data Gathering/thesis_data/Actual Data Run/Stage B/US_full_period_stageB_combined.csv
  Parquet : /Users/adam/Desktop/PCRA Index Thesis/Data Gathering/thesis_data/Actual Data Run/Stage B/US_full_period_stageB_combined.parquet


**Phase 2 — production run.** Article scoring + novelty weights over the full corpus.

In [41]:
from pathlib import Path
from datetime import datetime, date
import json
import numpy as np
import pandas as pd

#  Output folder 

if "FULL_YEAR_PHASE2_DIR_US" not in globals():
    _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    FULL_YEAR_PHASE2_DIR_US = (THESIS_DATA_ROOT
                                / f"full_period_US_phase2_{_ts}")
    FULL_YEAR_PHASE2_DIR_US.mkdir(parents=True, exist_ok=True)
    print(f"✓ New Phase 2 folder: {FULL_YEAR_PHASE2_DIR_US}")
else:
    print(f"✓ Using existing Phase 2 folder: {FULL_YEAR_PHASE2_DIR_US}")

#  Config guards 

if "RUN_COUNTRY"               not in globals(): RUN_COUNTRY               = "US"
if "RUN_START"                 not in globals(): RUN_START                 = date(2008, 1, 1)
if "RUN_END"                   not in globals(): RUN_END                   = date(2025, 12, 31)
if "ALPHA"                     not in globals(): ALPHA                     = 0.5
if "EMBEDDING_MODEL"           not in globals(): EMBEDDING_MODEL           = "all-MiniLM-L6-v2"
if "NOVELTY_WINDOW_DAYS"       not in globals(): NOVELTY_WINDOW_DAYS       = 7
if "USE_SENTENCE_TRANSFORMER_BETA" not in globals(): USE_SENTENCE_TRANSFORMER_BETA = True

#  NLP guards 

if "nlp" not in globals():
    print("Loading spaCy model...")
    nlp = load_spacy_model()
    nlp.max_length = 2_000_000
    print(f"  ✓ spaCy: {nlp.meta.get('name','?')}")
else:
    print(f"  ✓ spaCy already loaded: {nlp.meta.get('name','?')}")

if "hazard_matcher" not in globals():
    print("Building matchers...")
    hazard_matcher    = build_phrase_matcher(nlp, LEXICON_ALL,
                                              label="HAZARD", attr="LOWER")
    exclusion_matcher = build_phrase_matcher(nlp, TRANSITION_EXCLUSION_LEXICON,
                                              label="EXCLUSION", attr="LOWER")
    print(f"  ✓ Hazard matcher: {len(LEXICON_ALL):,} terms")
else:
    print(f"  ✓ Hazard matcher already built")

#  Input guard 

if "full_year_stageB" not in globals() or full_year_stageB.empty:
    raise RuntimeError(
        "full_year_stageB is missing or empty. Run the Stage B cell first.")

if "text" not in full_year_stageB.columns:
    raise RuntimeError("Stage B output has no 'text' column.")

#  Helpers 

def _safe_to_csv(df: pd.DataFrame, path: Path) -> Path:
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].astype(str)
        elif out[col].dtype == "object":
            out[col] = out[col].apply(
                lambda x: json.dumps(x, default=str)
                if isinstance(x, (dict, list, tuple, set)) else x)
    out.to_csv(path, index=False, encoding="utf-8-sig")
    return path

def _safe_to_parquet(df: pd.DataFrame, path: Path, label: str = ""):
    try:
        df.to_parquet(path, index=False)
        print(f"  ✓ Parquet saved: {label}")
    except Exception as e:
        print(f"  ⚠ Parquet skipped ({label}): {str(e)[:80]}")

def _drop_nonserializable(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[c for c in
        ["doc", "country_mentions", "diagnostic_taxonomy_terms"]
        if c in df.columns], errors="ignore").copy()

def _ensure_story_id(df: pd.DataFrame, prefix: str = "row") -> pd.DataFrame:
    out = df.copy()
    if "story_id" not in out.columns:
        out["story_id"] = [f"{prefix}_{i}" for i in range(len(out))]
    out["story_id"] = out["story_id"].fillna("").astype(str)
    blank = out["story_id"].str.strip() == ""
    if blank.any():
        out.loc[blank, "story_id"] = [f"{prefix}_{i}" for i in out.index[blank]]
    if out["story_id"].duplicated().any():
        out["story_id"] = (out["story_id"].astype(str) + "_"
                           + out.groupby("story_id").cumcount().astype(str))
    return out

def _normalize_pipeline_dates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "assigned_date" not in out.columns:
        for col in ("publish_dt_utc", "publish_dt", "market_day"):
            if col in out.columns:
                out["assigned_date"] = out[col]; break
        else:
            out["assigned_date"] = pd.Timestamp(RUN_START)
    out["assigned_date"] = pd.to_datetime(out["assigned_date"], errors="coerce")
    out.loc[out["assigned_date"].isna(), "assigned_date"] = pd.Timestamp(RUN_START)
    out["assigned_date"] = out["assigned_date"].dt.normalize()
    return out

def _compute_tfidf_beta(df: pd.DataFrame, window_days: int = 7) -> pd.Series:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    work = df.copy().sort_values(["country", "assigned_date"])
    beta = pd.Series(1.0, index=work.index, dtype=float)
    for cc, sub in work.groupby("country", sort=False):
        sub = sub.sort_values("assigned_date")
        if len(sub) <= 1:
            continue
        texts = sub["text"].fillna("").astype(str).tolist()
        try:
            X = TfidfVectorizer(lowercase=True, stop_words="english",
                                max_features=50_000,
                                ngram_range=(1, 2)).fit_transform(texts)
        except Exception:
            continue
        dates = pd.to_datetime(sub["assigned_date"]).to_numpy()
        win   = np.timedelta64(window_days, "D")
        for pos, idx in enumerate(sub.index):
            prior = [j for j in range(pos)
                     if dates[j] < dates[pos] and dates[j] >= dates[pos] - win]
            if prior:
                beta.loc[idx] = max(0.0, 1.0 - float(
                    cosine_similarity(X[pos], X[prior]).ravel().max()))
    return beta.reindex(df.index).fillna(1.0)

def _safe_compute_beta(df: pd.DataFrame) -> pd.Series:
    if (USE_SENTENCE_TRANSFORMER_BETA
            and "compute_novelty_weights" in globals()
            and "compute_embeddings" in globals()):
        try:
            print("  Computing w_n with sentence-transformers...")
            return compute_novelty_weights(
                df, model_name=EMBEDDING_MODEL,
                window_days=NOVELTY_WINDOW_DAYS).reindex(df.index).fillna(1.0)
        except Exception as e:
            print(f"  ⚠ ST beta failed: {e}. Using TF-IDF fallback.")
    print("  Computing w_n with TF-IDF fallback...")
    return _compute_tfidf_beta(df, window_days=NOVELTY_WINDOW_DAYS)

#  Prepare Phase 2 input 

print("\n" + "═" * 100)
print("PHASE 2 — ARTICLE SCORING")
print("═" * 100)

phase2_input = full_year_stageB.copy()
phase2_input = _ensure_story_id(phase2_input, "US_phase2")
phase2_input = _normalize_pipeline_dates(phase2_input)

if "country" not in phase2_input.columns:
    phase2_input["country"] = RUN_COUNTRY

phase2_input["c_o"] = 1.0

print(f"  Input articles : {len(phase2_input):,}")
print(f"  Date range     : {phase2_input['assigned_date'].min().date()} "
      f"→ {phase2_input['assigned_date'].max().date()}")
print(f"  c_o            : 1.0 (uniform — no credibility weighting)")

#  Score corpus 

print("\nScoring corpus...")

_score_cols = ["story_id", "country", "text", "c_o", "assigned_date"]
for _opt in ["publish_dt", "publish_dt_utc", "media_name",
             "category", "title", "url"]:
    if _opt in phase2_input.columns and _opt not in _score_cols:
        _score_cols.append(_opt)

scored_df, hits_df = score_corpus(
    phase2_input[_score_cols].copy(),
    nlp=nlp,
    hazard_matcher=hazard_matcher,
    alpha=ALPHA,
    aggregator="multiplicative")

if scored_df.empty:
    raise RuntimeError(
        "Phase 2 returned no scored articles. Check score_corpus inputs.")

scored_df = _normalize_pipeline_dates(scored_df)
print(f"  ✓ Scored : {len(scored_df):,}")
print(f"  ✓ Hits   : {len(hits_df):,}")

#  Novelty weights w_n 

print("\nComputing novelty weights (w_n)...")

_beta_input = scored_df.merge(
    phase2_input[["story_id", "text"]], on="story_id", how="left")
if "country" not in _beta_input.columns:
    _beta_input["country"] = RUN_COUNTRY
_beta_input = _normalize_pipeline_dates(_beta_input)

scored_df["beta"] = (_safe_compute_beta(_beta_input)
                     .reindex(_beta_input.index).values)
scored_df["beta"] = (pd.to_numeric(scored_df["beta"], errors="coerce")
                     .fillna(1.0).clip(0.0, 1.0))
scored_df["weighted_score"] = (scored_df["score"].fillna(0.0)
                                * scored_df["beta"].fillna(1.0))

print(f"  ✓ w_n mean : {scored_df['beta'].mean():.4f}")
print(f"  ✓ w_n min  : {scored_df['beta'].min():.4f}")

#  Merge Stage B metadata back 

_meta_cols = [c for c in full_year_stageB.columns
              if c not in scored_df.columns and c != "text"]
scored_full = scored_df.merge(
    full_year_stageB[["story_id", *_meta_cols]],
    on="story_id", how="left")

#  Save outputs 

_scored_csv     = FULL_YEAR_PHASE2_DIR_US / "US_full_period_phase2_scored.csv"
_hits_csv       = FULL_YEAR_PHASE2_DIR_US / "US_full_period_phase2_hits.csv"
_scored_parquet = FULL_YEAR_PHASE2_DIR_US / "US_full_period_phase2_scored.parquet"
_hits_parquet   = FULL_YEAR_PHASE2_DIR_US / "US_full_period_phase2_hits.parquet"

_safe_to_csv(_drop_nonserializable(scored_full), _scored_csv)
_safe_to_csv(_drop_nonserializable(hits_df),     _hits_csv)
_safe_to_parquet(_drop_nonserializable(scored_full), _scored_parquet, "scored")
_safe_to_parquet(_drop_nonserializable(hits_df),     _hits_parquet,   "hits")

print(f"\n✓ Phase 2 complete")
print(f"  Scored articles : {len(scored_full):,}")
print(f"  Hits            : {len(hits_df):,}")
print(f"  Score mean      : {scored_full['score'].mean():.6f}")
print(f"  Weighted mean   : {scored_full['weighted_score'].mean():.6f}")
print(f"  Output folder   : {FULL_YEAR_PHASE2_DIR_US}")

display(scored_full[["story_id", "assigned_date", "score",
                      "beta", "weighted_score"]].head(10))

2026-08-11 20:41:09,239 | INFO | === PHASE 2 | scoring 41920 articles (aggregator='multiplicative', α=0.5) ===


✓ New Phase 2 folder: /Users/adam/Desktop/PCRA Index Thesis/Data Gathering/thesis_data/full_period_US_phase2_20260811_204109
  ✓ spaCy already loaded: core_web_trf
  ✓ Hazard matcher already built

════════════════════════════════════════════════════════════════════════════════════════════════════
PHASE 2 — ARTICLE SCORING
════════════════════════════════════════════════════════════════════════════════════════════════════
  Input articles : 41,920
  Date range     : 2008-01-01 → 2025-12-31
  c_o            : 1.0 (uniform — no credibility weighting)

Scoring corpus...


2026-08-12 15:39:29,802 | INFO |   scored: 41920 articles, 410971 total hits (291 unique terms)
2026-08-12 15:39:29,812 | INFO |   score distribution: mean=0.0043, median=0.0000, max=0.7333, n_zero=29541


  ✓ Scored : 41,920
  ✓ Hits   : 410,971

Computing novelty weights (w_n)...
  Computing w_n with sentence-transformers...


2026-08-12 15:39:31,252 | INFO | Loading embedding model: all-MiniLM-L6-v2
2026-08-12 15:39:31,300 | INFO | No device provided, using mps
2026-08-12 15:39:31,635 | INFO | Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


  ✓ w_n mean : 0.3699
  ✓ w_n min  : 0.0000
  ✓ Parquet saved: scored
  ✓ Parquet saved: hits

✓ Phase 2 complete
  Scored articles : 41,920
  Hits            : 410,971
  Score mean      : 0.004276
  Weighted mean   : 0.001310
  Output folder   : /Users/adam/Desktop/PCRA Index Thesis/Data Gathering/thesis_data/full_period_US_phase2_20260811_204109


,story_id,assigned_date,score,beta,weighted_score
0,78d9165e7770cc24a4a99b2c50ef1cda49d156606b8ed0...,2008-01-01,0.000000,1.000000,0.000000
1,53c704e3ff750b03b11cf02a5a47684ff05bf89294d4be...,2008-01-22,0.000000,1.000000,0.000000
2,bb698d0704d3c7a558913f4ff504a41c400ae8a13455a8...,2008-01-28,0.000000,0.927722,0.000000
3,fe35382e404618c5b7ed6bf92888f955b0966bcb24dbca...,2008-01-29,0.000000,0.691225,0.000000
4,bb327bbe08a29157b14ed91b22b59ffe19d02da0b50946...,2008-01-30,0.000331,0.480771,0.000159
5,8af61ac1ac960d26ed7068393af003408ce063482bec6e...,2008-01-30,0.000289,0.545572,0.000158
6,4b1e4be686afdee0194dbb21fa5b6e1436e4634811bcb3...,2008-02-06,0.001313,0.623014,0.000818
7,5f7969cdee9add1411e7171ca0bbb83e9bd0972858c8f9...,2008-03-04,0.000000,1.000000,0.000000
8,8bb2234b3fb1f52d36117d10fce9e48525a067e1072e2a...,2008-03-04,0.000185,1.000000,0.000185
9,65088cc23121df63c54bdfcb5f411a7ecfde17a84a0d90...,2008-03-24,0.000048,1.000000,0.000048


**Phase 3 — production run (final).** 

Daily SSI & CSP indices — BIC-selected AR(0–5), no NaN-filling. 

**This is the cell that generated the reported thesis results.**

In [49]:

from pathlib import Path
from datetime import datetime, date
import json
import numpy as np
import pandas as pd
from statsmodels.tsa.ar_model import AutoReg

#  Output folder 

if "FULL_YEAR_PHASE3_DIR_US" not in globals():
    _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    FULL_YEAR_PHASE3_DIR_US = (THESIS_DATA_ROOT
                                / f"full_period_US_phase3_{_ts}")
    FULL_YEAR_PHASE3_DIR_US.mkdir(parents=True, exist_ok=True)
    print(f"✓ New Phase 3 folder: {FULL_YEAR_PHASE3_DIR_US}")
else:
    print(f"✓ Using existing Phase 3 folder: {FULL_YEAR_PHASE3_DIR_US}")

#  Config guards 

if "RUN_COUNTRY"            not in globals(): RUN_COUNTRY            = "US"
if "RUN_START"              not in globals(): RUN_START              = date(2008, 1, 1)
if "RUN_END"                not in globals(): RUN_END                = date(2025, 12, 31)
if "CSP_ROLLING_STD_WINDOW" not in globals(): CSP_ROLLING_STD_WINDOW = 252
if "SSI_LAMBDA"             not in globals(): SSI_LAMBDA             = float(np.exp(-np.log(2) / 30))
if "AR_MAX_LAGS"            not in globals(): AR_MAX_LAGS            = 5

if "scored_full" not in globals() or scored_full.empty:
    raise RuntimeError("scored_full is missing or empty. Run Phase 2 first.")

#  Helpers 

def _safe_to_csv(df: pd.DataFrame, path: Path) -> Path:
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].astype(str)
        elif out[col].dtype == "object":
            out[col] = out[col].apply(
                lambda x: json.dumps(x, default=str)
                if isinstance(x, (dict, list, tuple, set)) else x)
    out.to_csv(path, index=False, encoding="utf-8-sig")
    return path

def _safe_to_parquet(df: pd.DataFrame, path: Path, label: str = ""):
    try:
        df.to_parquet(path, index=False); print(f"  ✓ Parquet saved: {label}")
    except Exception as e:
        print(f"  ⚠ Parquet skipped ({label}): {str(e)[:80]}")

def _normalize_pipeline_dates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "assigned_date" not in out.columns:
        for col in ("publish_dt_utc", "publish_dt", "market_day"):
            if col in out.columns:
                out["assigned_date"] = out[col]; break
        else:
            out["assigned_date"] = pd.Timestamp(RUN_START)
    out["assigned_date"] = pd.to_datetime(out["assigned_date"], errors="coerce")
    out.loc[out["assigned_date"].isna(), "assigned_date"] = pd.Timestamp(RUN_START)
    out["assigned_date"] = out["assigned_date"].dt.normalize()
    return out

def _build_ssi(s_daily: pd.Series, lam: float = SSI_LAMBDA) -> pd.Series:
    """SSI_t = lam * SSI_{t-1} + (1-lam) * S_t,   SSI_0 = S_0."""
    x = s_daily.values.astype(float)
    out = np.empty_like(x)
    out[0] = x[0]
    for i in range(1, len(x)):
        out[i] = lam * out[i-1] + (1.0 - lam) * x[i]
    return pd.Series(out, index=s_daily.index, name="SSI")

def _build_csp(s_daily: pd.Series, ssi: pd.Series,
               max_p: int = AR_MAX_LAGS,
               roll: int = CSP_ROLLING_STD_WINDOW):

    s_tilde = (s_daily - ssi.shift(1)).dropna()

    # p = 0 baseline: residual = demeaned s_tilde
    best = {"bic": np.inf, "p": 0,
            "resid": s_tilde - s_tilde.mean()}
    n0 = len(best["resid"])
    sigma2_0 = float((best["resid"] ** 2).mean())
    if n0 > 0 and sigma2_0 > 0:
        best["bic"] = n0 * np.log(sigma2_0) + np.log(n0)

    for p in range(1, max_p + 1):
        try:
            m = AutoReg(s_tilde.values, lags=p, old_names=False).fit()
            if float(m.bic) < best["bic"]:
                best = {"bic": float(m.bic), "p": p,
                        "resid": pd.Series(m.resid, index=s_tilde.index[p:])}
        except Exception:
            continue

    resid = best["resid"]
    roll_std = resid.rolling(roll, min_periods=roll).std()
    csp = (resid / roll_std).rename("csp")
    return csp, best["p"], best["bic"]

#  Prepare Phase 3 input 

print("\n" + "═" * 100); print("PHASE 3 — DAILY INDICES"); print("═" * 100)

phase3_input = _normalize_pipeline_dates(scored_full)
if "country" not in phase3_input.columns:
    phase3_input["country"] = RUN_COUNTRY
phase3_input["score"] = pd.to_numeric(phase3_input["score"], errors="coerce").fillna(0.0)
phase3_input["beta"]  = pd.to_numeric(phase3_input.get("beta", 1.0),
                                       errors="coerce").fillna(1.0).clip(0.0, 1.0)

print(f"  Input articles : {len(phase3_input):,}")
print(f"  Countries      : {sorted(phase3_input['country'].unique())}")
print(f"  Date range     : {phase3_input['assigned_date'].min().date()} "
      f"→ {phase3_input['assigned_date'].max().date()}")

#  Build indices per country 

phase3_daily_frames, phase3_country_outputs, phase3_summary_rows = [], {}, []

for cc, sub in phase3_input.groupby("country", sort=False):
    sub = sub.copy().sort_values("assigned_date")
    print(f"\n  [{cc}] {len(sub):,} articles "
          f"({sub['assigned_date'].min().date()} → {sub['assigned_date'].max().date()})")

    # 1. S_daily on a complete business-day calendar (no articles -> 0)
    S_raw = compute_daily_score(sub)
    S_raw.index = pd.to_datetime(S_raw.index)
    S_raw = S_raw.sort_index()
    full_idx = pd.date_range(S_raw.index.min(), S_raw.index.max(), freq="B")
    S_daily = S_raw.reindex(full_idx).fillna(0.0)

    # 2. SSI: spec-compliant EWMA, init at S_0
    SSI = _build_ssi(S_daily, lam=SSI_LAMBDA)

    # 3. CSP: AR(p)-by-BIC residual/rolling-252 std, no fillna(0)
    CSP, ar_p, ar_bic = _build_csp(S_daily, SSI,
                                    max_p=AR_MAX_LAGS,
                                    roll=CSP_ROLLING_STD_WINDOW)

    daily = pd.DataFrame(index=full_idx)
    daily.index.name     = "date"
    daily["country"]     = cc
    daily["S_daily_raw"] = S_raw.reindex(full_idx)          # NaN on no-article days
    daily["S_daily"]     = S_daily                          # 0-filled for SSI/CSP
    daily["SSI"]         = SSI
    daily["CSP"]         = CSP.reindex(full_idx)
    daily = daily.reset_index()

    phase3_daily_frames.append(daily)
    phase3_country_outputs[cc] = {"S_daily": S_daily, "SSI": SSI, "CSP": CSP, "daily": daily}

    # Diagnostics inline
    csp_clean = daily["CSP"].dropna()
    rho1 = (float(np.corrcoef(csp_clean.values[:-1], csp_clean.values[1:])[0, 1])
            if len(csp_clean) > 2 else float("nan"))

    phase3_summary_rows.append({
        "country":         cc,
        "period":          f"{RUN_START} → {RUN_END}",
        "n_articles":      len(sub),
        "n_business_days": len(daily),
        "n_csp_obs":       int(csp_clean.size),
        "AR_order_p":      ar_p,
        "AR_BIC":          ar_bic,
        "CSP_rho_1":       rho1,
        "SSI_lambda":      SSI_LAMBDA,
        "S_daily_mean":    float(daily["S_daily"].mean()),
        "SSI_last":        float(daily["SSI"].dropna().iloc[-1]),
        "CSP_mean":        float(csp_clean.mean())   if csp_clean.size else float("nan"),
        "CSP_std":         float(csp_clean.std())    if csp_clean.size > 1 else float("nan"),
        "CSP_max_abs":     float(csp_clean.abs().max()) if csp_clean.size else float("nan"),
    })

    print(f"    S_daily obs       : {daily['S_daily_raw'].notna().sum():,}")
    print(f"    SSI obs           : {daily['SSI'].notna().sum():,}")
    print(f"    CSP obs (post-burn): {int(csp_clean.size):,}")
    print(f"    AR order (BIC)    : p = {ar_p}")
    print(f"    CSP ρ(1)          : {rho1:+.4f}   (should be ≈ 0)")
    print(f"    CSP mean / std    : {csp_clean.mean():+.4f} / {csp_clean.std():.4f}")
    print(f"    CSP max |·|       : {csp_clean.abs().max():.2f}")

#  Combine & save 

phase3_daily_indices = pd.concat(phase3_daily_frames, ignore_index=True, sort=False)
phase3_summary       = pd.DataFrame(phase3_summary_rows)

_daily_csv     = FULL_YEAR_PHASE3_DIR_US / "US_full_period_phase3_daily_indices.csv"
_summary_csv   = FULL_YEAR_PHASE3_DIR_US / "US_full_period_phase3_summary.csv"
_daily_parquet = FULL_YEAR_PHASE3_DIR_US / "US_full_period_phase3_daily_indices.parquet"

_safe_to_csv(phase3_daily_indices, _daily_csv)
_safe_to_csv(phase3_summary,       _summary_csv)
_safe_to_parquet(phase3_daily_indices, _daily_parquet, "Phase 3 daily")

print(f"\n✓ Phase 3 complete")
print(f"  Daily rows    : {len(phase3_daily_indices):,}")
print(f"  Output folder : {FULL_YEAR_PHASE3_DIR_US}")
print(f"  CSV           : {_daily_csv}")
print(f"  Parquet       : {_daily_parquet}")

display(phase3_summary)
display(phase3_daily_indices.head(10))
display(phase3_daily_indices.tail(10))

✓ Using existing Phase 3 folder: /Users/adam/Desktop/PCRA Index Thesis/Data Gathering/thesis_data/full_period_US_phase3_20260812_154725

════════════════════════════════════════════════════════════════════════════════════════════════════
PHASE 3 — DAILY INDICES
════════════════════════════════════════════════════════════════════════════════════════════════════
  Input articles : 41,920
  Countries      : ['US']
  Date range     : 2008-01-01 → 2025-12-31

  [US] 41,920 articles (2008-01-01 → 2025-12-31)
    S_daily obs       : 4,110
    SSI obs           : 4,697
    CSP obs (post-burn): 4,445
    AR order (BIC)    : p = 0
    CSP ρ(1)          : +0.0373   (should be ≈ 0)
    CSP mean / std    : -0.0028 / 0.9956
    CSP max |·|       : 14.54
  ✓ Parquet saved: Phase 3 daily

✓ Phase 3 complete
  Daily rows    : 4,697
  Output folder : /Users/adam/Desktop/PCRA Index Thesis/Data Gathering/thesis_data/full_period_US_phase3_20260812_154725
  CSV           : /Users/adam/Desktop/PCRA Index The

,country,period,n_articles,n_business_days,n_csp_obs,AR_order_p,AR_BIC,CSP_rho_1,SSI_lambda,S_daily_mean,SSI_last,CSP_mean,CSP_std,CSP_max_abs
0,US,2008-01-01 → 2025-12-31,41920,4697,4445,0,-45005.267984,0.037343,0.97716,0.003203,0.001946,-0.00277,0.995555,14.542452


,date,country,S_daily_raw,S_daily,SSI,CSP
0,2008-01-01,US,0.0,0.0,0.0,NaN
1,2008-01-02,US,NaN,0.0,0.0,NaN
2,2008-01-03,US,NaN,0.0,0.0,NaN
3,2008-01-04,US,NaN,0.0,0.0,NaN
4,2008-01-07,US,NaN,0.0,0.0,NaN
5,2008-01-08,US,NaN,0.0,0.0,NaN
6,2008-01-09,US,NaN,0.0,0.0,NaN
7,2008-01-10,US,NaN,0.0,0.0,NaN
8,2008-01-11,US,NaN,0.0,0.0,NaN
9,2008-01-14,US,NaN,0.0,0.0,NaN


,date,country,S_daily_raw,S_daily,SSI,CSP
4687,2025-12-18,US,0.000856,0.000856,0.002102,-0.567116
4688,2025-12-19,US,0.000008,0.000008,0.002054,-0.926971
4689,2025-12-22,US,0.000927,0.000927,0.002029,-0.503564
4690,2025-12-23,US,0.001726,0.001726,0.002022,-0.141347
4691,2025-12-24,US,0.002234,0.002234,0.002027,0.085759
4692,2025-12-25,US,0.000066,0.000066,0.001982,-0.874306
4693,2025-12-26,US,0.002849,0.002849,0.002002,0.376867
4694,2025-12-29,US,0.001588,0.001588,0.001992,-0.191691
4695,2025-12-30,US,0.001149,0.001149,0.001973,-0.382375
4696,2025-12-31,US,0.000791,0.000791,0.001946,-0.532881
